# NB11 · The end-visual: GPU pictures for Paper B (vinyl alcohol → acetaldehyde)
**Fruitful PES Explorer, notebook 11.** NB10 showed that the menu agent finds the barrier with a dozen Skala calls where classical rules do not. This notebook makes the **pictures that show why**, and the pictures that only a real Skala run can give. It ends with a four-panel poster, an animation and a zip. **Needs a GPU (L4 or T4); plan for about 2 hours.** Everything saves to Drive and every long cell resumes after a disconnect.

**What it does, in the order it runs (the plan page has the same steps drawn out):**

| step | where | what you get | time (L4) |
|---|---|---|---|
| 0 | CPU | NB10 results unpacked from inside this notebook, Skala smoke test | 2 min |
| 1 | **GPU** | **The hydrogen in flight:** Skala electron density at 9 geometries along the cheapest path, and two bond meters (O–H fades, C–H grows) | 5 min |
| 2 | **GPU** | **Relaxed truth check:** xTB relaxes every atom along the hydrogen jump, Skala re-computes the energies; a relaxed barrier next to the rigid-grid one | 10 min |
| 3 | **GPU** | **Microscope on the pass:** Skala re-scan at about 3.5 times finer spacing around the pass and the enol well (time boxed, resumable) | 60–70 min |
| 4 | CPU | **Replay:** the agent's map rebuilt pick by pick against the best classical rule (GIF) | 3 min |
| 5 | CPU | **Anchor-first control:** a hand-made "pass, then both wells, then refine" rule, 8 starts | 3 min |
| 6 | CPU | **Poster** (panels A–D), each panel as its own image, the zip (also copied to Drive) | 2 min |

**How to run:** Runtime → Change runtime type → GPU, then run the cells in order. Cell 6 prints seconds per Skala point first. Steps 1 and 2 come before the long scan on purpose: if time runs short you already have the most valuable pictures. No `GEMINI_API_KEY` is needed: no LLM calls are made here; all agent runs are the stored NB10 runs.

**What is honest about it.**
- The relaxed check is *xTB-relaxed geometries with Skala energies* (a standard, cheap approach), not a full Skala transition-state search. The caption on the poster says so.
- The grid barrier (75) uses a fixed recipe for the other atoms; the literature barrier is about 55 (from memory, unverified). Step 2 shows how much of the gap is the recipe.
- Density pictures come from the Skala SCF density of the same calculation that gives the energy.
- Time per point for 7 atoms was 6.0 s on an L4 in NB10; this notebook's estimates scale from that.
- **Dry run.** Cell 6 has a `SKALA_STAND_IN` switch (PBE/STO-3G on CPU replaces Skala) and the settings cell shrinks the scans, so the whole notebook can be tested in minutes. **Never report numbers from a stand-in run.**


In [ ]:
#@title 1. Install (3-6 min). Runtime -> Change runtime type -> GPU first.
import subprocess, sys, importlib
def pip(*a):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], capture_output=True, text=True)
    print(("ok   " if r.returncode == 0 else "FAIL ") + " ".join(a));
    if r.returncode: print(r.stderr[-1500:])
    return r.returncode == 0

pip("tblite", "scikit-learn", "scipy", "matplotlib", "pandas", "pillow")
# Skala: GPU wheel for CUDA 12 (Colab default), pinned to the version verified on Colab (2026.4 gave wrong forces in the
# optimizer notebook). Falls back to the CPU package if the GPU wheel is unavailable. Restart the session after changing it.
SKALA_VERSION = "2026.9"
if not pip(f"skala-cuda12x=={SKALA_VERSION}"):
    pip(f"skala=={SKALA_VERSION}") or pip("skala")
# gpu4pyscf is what makes PBE/B3LYP fast on the GPU; harmless if the Skala wheel already pulled it.
pip("gpu4pyscf-cuda12x", "cupy-cuda12x")

for m in ["pyscf", "skala", "tblite", "sklearn", "anthropic"]:
    try: importlib.import_module(m); print("import ok:", m)
    except Exception as e: print("import FAILED:", m, repr(e)[:120])
try:
    import gpu4pyscf; print("gpu4pyscf available: GPU DFT on")
except Exception as e:
    print("gpu4pyscf NOT available, DFT will run on CPU:", repr(e)[:100])


In [ ]:
#@title 2. Workspace on Google Drive, and the NB10 results unpacked (nothing to upload)
import os, json, time, gzip, base64, io, numpy as np
try:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKDIR = '/content/drive/MyDrive/fruitful-pes-vinyl-alcohol-visuals'
except Exception:
    WORKDIR = os.path.abspath('./fruitful-pes-vinyl-alcohol-visuals')
os.makedirs(WORKDIR, exist_ok=True); os.makedirs(f"{WORKDIR}/figures", exist_ok=True); os.makedirs(f"{WORKDIR}/data", exist_ok=True)
EMBED = {
    "ref_skala_def2-svp_19x13.npz": "H4sIAFXa9WoC/82ZC1jMWRvAp3KppqZUFGp3ioyW2m6ISoNUtDKuuYTJNN3VqFQUckstalbKLYm1CmGQ224Y2k1KiU3SJuOWy7ZdVkTC9/7PO7Of/s83z+z37ffss+Pp+T3nnPd23vc9p2PieWv0sGbgx5wx88USx4/yT0+GEWOiTYRouRpDn+GkiSIKZvjMmsKbq8aIZSRwAoTRgijOGDbHJdCJM5zNCYyMionyj1gUGRUgpOY9/MOjhTAfHewvEsJ4qN3o4Ww7B6vh7JXs//Gj3fFwf4CTtUBa6VX+vMNGIOUMndYQZiuQfuiaGHjKTiBlrcpcd8leIC2tLtkb5CCQntmZ77wTWDrqzeRYoOMex9dfAr3PJc5/BnKLW6bfKABqb5FZrAaeWeg6eAFQs/h2mgjsG5UaaqiD/Wlh7kdHgf1F/KF+I2D9gcfVuFdAAxcLo1Cwp781rCML+MWxqgNiYFdJSt8IapwZvJsLPHYj844BsH72Co/noKevnpr4E7BmuUVy/y8FUpsxbqPSwE/pdVHYbvAzfN0Q2xBYXxIZu5cBeg+jZunygdrly5PTgMNvvzCk9mWVcXDeFuAq54VfLgMuDXc56QtM/Npe0xG4benJgltgR2KdIxCCnx8HVhytAD9lhRy7EvCjLozZGwvrUfPvbOkAsvLWxk0Bvdxxxjqrgbv6BU3PBM6oiSneAVxvWHD1a2DZtLy8CUCbaZ2s66CXUngnxhHYsHKwXij40W861nQL/ARfyKu5Bn4e6httXwLroa8yfB8Ad8YmXrMHfR9rM+sw4OExR1jRQPXwvdGTgIcS7Y61g1zJ7UMRVF1OLpmU+pqqr/BG0mJg5+wiqQ/46VXZea8Q/Fxau3T6aZgvufurvy/I/64ZOaQQmL19hVgGzPWtLb0PjB/aMvM00Kl1YmMw8Ofstyk9gEUde5emg/6tGV799YAGiZfqU8Fuo6Wt32voAz2LwPrxMF6VMyHZCdb3qc9uqaDkiiv8WKCfdbl1Y18gJ2VrRwfM90jZFnoB2Bzk7LoEuHldsKcu8LZote5+sNNrU06bHfCaYcHBy7CPtnGHqql+8zjlXx4LYx+b5g3jYN3wcOqwcuDTiwOkLKovWoYdNwDKNGMqm2B+85XRZRLg9yH6CUFAxxFLC42ATZteaxSBnYOnOsv5QFZCqXov4GEZM4QHfqbu6qHxHhh2N2DrUpgP5y54ex7Iv5dm/xBo2iXkNwG5P/SPqwEK4lcl5wMdHAM+hAMZbeJLXwAvFvC8n4Cdw2duReYA09w+hPsCY1MvPq0Hjj/G50aD3LLxzw5eAw489z7jEXBk1oAHVyn76nllKcA7p1cMcAHO9CvzagK9yReSB+0Dtv7ufVYIXNx3crMDUMfT+d1HOP8Jp1YZVwBr1iT02AN6JVO8V3vDvpfJTD2LgfutLXdoQp6a3lxuNQYaPc7nvoD56wFarYnAw14e8fdAz7ne6qAVtf8l33utBvuNZwvHt4PdK+4J7B3A6pZV52cNFkgXuN/LpuoROffxZB2wt/FOxKv9wIN3n93Rhron9nbJ8QR2fXSunAtcyGiYMwa45lzXy7sg5yfQPWwDDI4eYOIOdiyal1m4gt/hPLcjXuBXffzY9M3gL/LzFXFZsH7ssuPlBJBfq3E0wAHsfJGdnX4ROMd3MXMgnA/N66kZ1DlxWXWpiDqvaT/vsh0GvPGhK78W5LYnM86OBY4bvbLAG+x8y2QG6oNdByu/USXg79zyg2Wx1H5E6WpCWN+QsreJutfOnLWvaAQO/8bTiLq/akMD1/xE+TPnWf0KNHi53vQhMEa8574E+GL3/sXzgIKI3+yvUvpzj5dWgb3xDsL0pWC/5sqApinAfOu2YZNgvnSjRr4eyBWqR2t+B/SqZjWwQL/4ecXzg8Do2VpHtRzhfjHaJLMD3jxkXGYBDN9Tw3oC6yZd5py11H04NTuCukff6ly50Ah21w056uEMfkw7QjZ/BePkvac4lrAefSjOg7pnj1V0BdmB3tCwET1igA2ttVe2A91r29W7qN8TPRedHwl+xBMSDIcB1c3f5zTCfPj5H67GA2+dfGtWR91znlcce1L95fl+gj346Zc6JmMwzE8oEm+oAf6Q17KTqsu8oc83pgJn87bNPgC89uTDWmp/KQNFD6h7eyjc48Tv2sz1DPBXNMampYja33rGQEq/c9uI2znQH83Prt39DfxN4siOU/dU+8izO6qBZk4rtah7Pq1p5kRqH/ubnW/nAS/UXKncBWRNHpC+Ejg21OfbOdT9ur5w2yqqftqBx3OBXlmJHYFAttv+hJfQD6fN778IBj82v+1sPAH7SnOX6VP35wmBNZPy8/RGTnkOsPnRygvfAvdkutWtAf5Y13aI6j/1632zjYD5O14b3wO9yUG7vc8Def3WR2rCPK/bcyMzOjlI8dzQheeGIDIiVhgVJAxQPDvi1FBUwf/47EhcbPd3PTvU/lEfxt/lqHvVPtRUPlVUTQuqFhESI4z67x6KIX/bQ7GPPGyWnPKwGAZy6srJpFFLTh0aFevaSuRYNLuKeX059Wj6TBX+9WhjFo30uFhK4tSm7V/ZPnRp8TJp83R7vVXsR1tJfuhy/VTY0VUy1lTihy6vR5un509PSXw6SuSYf3Lf9Hor/Pf5k/Z1ldSljxJ7qvJC758+NJop6QdV+kwV/alqf3pK4mWqOIf6tPNM96MYT6XFpa3EH91+HxX9Rc9/AM2/og6GSvJEP0c6KvpeX0m8Wkri1VZiT0tFXXRocdP90Pu9t4pzQLerqWQfujS/Wir8aCnpH00l55B+zg2V3G9MJXFqKznPyvKso6KPVN3fWkrsa6voBx0lcdPzSM+zFs0uS0n+mErqpqPk98qf3Y/CX/cnR8pUWZTiycGEJ0e0EJ6KAdH/1G+nIATuPvFALjCpM9iEIrfXSQMuuYjukHl2vJ8xRV7Dnr4UW5duJpSuO25IUdaD34+i+PM1hLLBG8i6+GWGEUVJZRWRyw1/R9ZbrxYQP/rO6cRu7sNiMs9rdxpA4rjp3p/I5RQR/dYpEvRbfwT9NuUTssvuE7uM3dpEXpYpJPLsE+pkzDApJ35sb14m9hl1vxC9qjYxmWek15Ax78V9oicLr0b9DBnZf+6Qkyjve5PMV33WQPxJnVaQuKt6upP8tJ54g3ae12Ge6tsxbk0N1H+uTuxx056h/XN3Mf5hdbgfh3eYpyOVGGfqVhJfbvtDsq7vn0Tmz+3TZRM9SW/0d6mD0LamN1nnHi8h/iW928h8ku8JMma8NMY83epE/wVlRN72RgNZZ1c3YL1uTiD5l4a/IfNJox+TOGQjsG76+elYF4M8Mi8JNSbzuZ2/Yv23PSFkPywlcvzCd8Su7KMd2j/9OyH/JfZJ0sJY7Ae9a0Q+N8merDOGvSD+9WNiSH5513UwTh+0L649QeQlRkZkLHlaTtZb37QTSud/R/bBPTrSlOh5mGIf9D5F7OW2rcD6uzuTvEhsORjnGE2iz3etI+s8y5uErYIUjGPidsI5TLNBJA49W5JPbvwBrNe9IRinyJDExzUxwDzoXCd6+lpjMd5QLTIvtewietIHLkSPyzTH+hnUkXX2STPM57Z+RI790xKyL95CVxJvLgfjlZh/hvV3HUfGsgeOhOLLAqzPzEdEj7H/IsmHbeEg9Jfynvix5TjjuL55JLGXfADjD/LAc1c2EfvpJJvI2b4ejn011wf7yOow2Z/EyhPr7LoI4zgwGfN8ZyL20Vejif/a7GkuxF7cJhJX7hkuqQNvUpgZmd9kgf2x6CPeE9VOGMfGGeQcScaboX0zDu7PcjDm//F8rPtMUxIPnxuG5zSuGPfX6Yf10rpE1kWt0/H8CCIxn2G4nyo1b4z31Rw8l5Y+2FfrphD7tjsi8Tw4bsR6l2h8RuJfU4V928gneqKLU4kcXxCC9yJXSuLn9Z+HfRAfg3Y6YzC/OktwPyGzMG6HFSQv4pRI7O/XaXi/pDCxP5ZdJfnilm4xwv6Lx/Ox+CjxI6oWYh41k3Hd7j7RSzKZj3ViJuB9dnQS3lPJPcn++LfNkaa/4LkMziZ+bEebYx/qLsJzGhmB99X86Xi/83gYd4MN7scY85u0j0P0ku6J8PyPxjj5M4aa4P3Ygnqaj7FuVqfwPH4zDfMeuBDzPIiLvyc29lHcF1ivCVFkf+LNIegvBOUk4gi0Uy/B/pcaYp+0huLvnbJN2M+DvNB/eQ6Rl8z3RX31LJJHWdxYsn/+PRH+HkhagPW9cYTYlbz6ldgVRzeSuMTFm7FPfPGes/06C/vYl4v30HIP4k90KAHvcUEq3suir4i86EwuGTNYwbiegn3BiA3D/V0w6c/t/uh4Vcv3/PR7jphgYYy/4slhIv9GykTt//Pk+CvvDfmT45PPYC7SUc6xcrrLOUlOH2TSNPl4pnw8Wz6eKx/Pl48XIrl8+bw/UrpYvh4gXxfK1wNpKZ2S5Tr9078v/vG10TO5iIJ/LZ0OfzmdDuTzyC2QfBrddu2kPs/dKq5TnyY3jLJVPt/m5kjkX7pVkvV2tyCi91ou90Yu91Yu905up8uN562mbq3R/Q+w1CuW+un5SUGTSJuRP8fSVagvUanvR6kf3W4qkWCk21eqdFXqmzyFN61uqpbQUH98r0dXo17jCjVmN7Ud8N/wT97mdEXqRFEHhvrp7k93IOPf54uuRnXNM3lzdE8K35TBkIfYsxc11Qv+eYH11eR7jH8BHO4WxN0eAAA=",
    "ref_xtb_19x13.npz": "H4sIAFXa9WoC/+1Ze1iN2Rr/0sUIIZcj6syuFpmRQRoq1EYpjUtsRUp22hWV7tex01cRKumCjFuJSVGSXEomS6SLSxozmMKYXCpySSRhzLv3u5qRc/45Z84zzznPsXp63udd33pvv/fyrefbNl8pKo3icGlzY2J1fN+xpcwN4Cy+8PYNU+D6ckaf4JFOmjLbdpaNvQIXzK3Uk7gGuPjrmQj0JrkZ6ekL9Nx8/AP9nb2X+PhLXGX70529AlxhP2CZs68r8CPGGusLxo77TF8QLvg3l2qy4V2V3i6Epnn0TDQKJbSf+El70TpCB6q9rf8mldCwfPPv0/cSuqBD8jo2l9CzWi+OnjlE6Em3uru2eYSaDtFU2XqA0A7HygDxPkLFCvb3UvYQmuBk2xKUTuiW6MhchV2ELvz52Y05fqA/a+/ax9GEnvkxwOtOEqF3Wt41XNxJaLp6hLkE5DdJle8eBDtf5m/7PBzs6IpnmMcD1c46OeTRQUJXNodOjskhNDHTKqQ5m9Dbu3o+9AY5pd6pt0Z/S2jR+PpqL4hDN8rgjWIsoZaPBzh+vYXQF0FPjeeDPz90y+72Cs5bexj20gI7D5qM7OohjgHjy4oqwE7SCsuxy4FmPFa9Zwn7yb0rj52Fc2Pq6jMrIc6kybdS8jMJNfB4/EnvlYSeP5X706Z4Qj8Z2tYRsJXQmsl2i9+CHVOvKGUe7JQquVzpD/4aFYx90gT+7zGd8cIB9Dt7Z487L7NTYvFoNNC+9jXHc+DcuBjF9lrQ3xac3xaTRqj73rADw78GvMssbliCHX60jiQS7ExTahoTB3aaLfV6LILzY0r76Kfvh/iiDsfuBH/tg5fwRmBPPWvBc1fQOzQiVk2aBXh9Wr26NYPQqikb66bvILTxxYxTVzcRWnlsorl9MKGbXY6sM1hPaHHqLHE44PbUq05aAPlz112b5wR5TWo/kFUOOJuI7NQGgN36k3v03aA+5pSI7yntJjRkerupBehdqeVwOhjkt0hjqls3Evquyu1gH9DrPsrq0f0VhB7aX3zGgCe07O7nDk82ELquMf+KBM5/92rINZNtgNfJuFuDthNa0ktkpw183fWJVzOgHs8Z9NCpTSF076qtTweA3v4LNpxIA72i+ETRaairttN/qzweTujVOxmHTjhD/T48308D6i7O+1n4Q9jPLT/mr7UW6kVj7mJNwNOv4jt7R7C/WWG41VDg5wVuvCcFfSOnDtZvW0PoqB2/NJyNJPRm7fqbOiCfd0m7chrU18y1d1NEAYTa3ri73dOe0EeldUt1JYSWd+gvnQTxXZY6nA4IJHRHatXL2DBCu7vXR1hKCb1el/biJNAnGt6L7KB+TAf/NKMA9BlfyOzwDII6OvtsWi34W2bWHCHwJrQw27/VwYPQ1y+WKCg6Eap3bmb3EDdCV4hLjZu8CB19yPMXNTg/y/LISupPqGuqbqwZUAW/08/DfKEPHWwTVoOealvdGc1wfr+g/OelnlBPqlX9LZcTuugre5oH+qaNGKijAv7bVVxukYC9y1k15iPB/10LNIIPgJ9zfJqDqyBf2dkXL0wCnBcpnRdFRhF64twvivNWEbrtesF80xBCL45Qr2gBe6F9NKq5ZWBnTObQIaD3jPqrquilkN+GnQesIS+P4+IN0iBe62lTJ80BfH+ePPLXOOhXw33nRFNhLryc3dYUvZlQbsqjGAfIu81V9X0RUB9xm69rGiZDHMVD1cohT6bDtHUyIS8uNfqimRDv5vjyXWoQz9yXDaMaxITeMJ+uPR3wnh96TrQezuufuGYdC/UeqZehqwRzp9hzOV8N/ZBl0Sh8AnXtlF8Xnwp1HfrTZ3lqwH/xYJfJM+izdk3Xhc+h78QPJ/T7MQ5w2XYjdi/0pZbVkR7FgNfs+ad7PgCcHKfXah0EOzvi1/n1AX/DdYIubgX5ywezpIXQfz3PNCQoQl+O9u2n6wR9f61d8UI7zJmmb41qJ0LfLlIpfeYP5wYF2zn7g5z+m43GP0C8AqvZSzUjCPV8V2fZBvXUR9gSFQZzOqPfSw83iKdtZLbFJoinstzm4UToR0/NAePMQN/N15Wtbw5DPSz2kRoeJfR+fG7myCOEqo4LMQ0C+6MHe9UUwdy4rGCw3QBw+FxjUGk+9JfI6McVD6AuvapLSnPAju0wT4ujkI8ixxCHVJhLTs9Fof6A07NL3NRykD9lXX3BEOwtW2TQ9grsxaTMru12jNCd99S9m8Ce/vUFN9LAXsRG5RxtOO9Y3X3vcJgX33wxsagR6sCuY2Lp36Evjc9bG+eAncn9v1fOgfjzV6VmKwAezZp1JsYwx654nCnTBvxURIYeEUBzFqskPQC7RZN0lqkXEJqieufxHrB/zTGhwQSeS8qTZljCHH5nohoxF+pDpdsEHU2oX6eK8gvFEOdYW9fBG8BOWer2tNtgp3T7tbcjQZ+5X8cqHuS13W+mtwFvOq2xXjZHS96Mn+sA/vtqhWz7EnilVhtVS3huaZO/Wwjvo+MD+9maukN+niT0lUI93J2Zcv9EAsyH7spDV8McVapKLpTZ+SEkPOkoyNlMWa4nATsTNgzJ3A367i/w7l4CesJ3W1hHA75pJjM/XQhz18tvcKER4PUqbMDufZBfmy7XjUTR2KLO60ZvuG64+HgHu/q7u0o6rx0hCni0k/7Ta4d06di/6tqh8P+5umYtbIlFQGfWekDWvJcHuvr/axfF5X/ZRfHdx/VxfVwf18f1cX1c/zOr65Vjlr3t884nPeHKEeAKV0VJwH/r1ymZD5Y76szkzoSFyik1CUZeGiSn/MAwORVuCZBT8Zo1eC4HzwnfMrkv8bxR/xzc18F9YRXuJ1bno77THkhrJEiTUB/38mvkHZj9XMaXI39oWCbqO78e7amuxnM+q3DfOkFO07vj/sINB/C5eizq8dmOck9RH61Av2gt4z2j5bSw8luUm4L7fB7zJ7tTDv3iLFbicyl7nof6uIaN6EcrUlqA/o05m4g8h3JCR9Rjte848vERcuqrlSynl8UpqEcxG59HhiD+CjzaSXfD/VGIM3cP8TyiX418f7YfhOd4W5TnP0NKh6MfnNpaOa2w281wQrwoO/90N+4r+RTI6e3ySJRzX47UDfHmixGH0JYs3I9GPUIt9Lfx4D7M46+Ig3Ab2r9dvxnl06NQLpXhnMjw5xmfzKgK4iYMZ7wHwz+G0aOMJjK/SorxfBnml7+JcQmfIeXuM1wkDJeRjC9lelidU2/mz2r2XA3x5bsjfzsN60tYyOpCgPuk4pKc9n2IfcFHMtw9wjGf5li3FQ1Yd7wn1gvXIZVTad+jqL8N88S3Mb+HxGBfFWK9CB8xv6ww34KdGC83l+VpCeuXSei3YNU6lLvB+nQ16rt9lOmrZHG7YF3bXN2C8posvh7MjwJGyxjNCulSn7wY4+z1pAj317L6H49zhXdm/TRvGdqrYXNlGsM/DuV5EeoVJCMOgjT0n0ajX/wgFs+v2DfiCUnYP4+x/uhBnF9cMfYlJ8Y4aR9fpEqsLjTQv/TpOGfMJ56U02afQhnlC1IbUb4R+zU9idkfzvAvQb/F+ejnIZ801NuCdsT3EGf+e4xbcAV57gryfT0ykE/GPis8nsmeI57cBja34li9NrK548fmdRrrIwsXpCLET2CE8QgPo1+3RXtxXzOJ5W8FPndDvb41uD9mRCruO2Nc6d8hfuJeTF8uy9cO9p7IQ0rXs/4LwPryDdqPVHMP7r9kcScEYh1n4VxYqIF1Qm9hPGrDy+S0/fkhPK+FuIjXY/74TC/cz2c4ZOMcpJuRF/vhe4CeRBy4JJQ/u+kwyg/BPFKblV3mjvAUw1kQ3AV3Xo29l2oSWf7x/cVvZPMhA2nf+TgHhQtYnnIZPo6oJzCXzf21iCPP3kf8cMwbf5jV/Vn2Hmll8lFrzLpeOl5cF1u+/50jcJlroHPnlWMw+yI1WOE/c+X4M/eNzivHH4sIkRoyasqoOaMzGJ2NlJ/L+PmMt2O8PeMdGO+EVChm+85I6VL2XMKeu7LnbsIP7nFbJs97//fF3z8bNbIjnfTPwTnuT8M5Tr7umLnJ132zbd/IVpPZxQuy1YyVyz1l+y1mhvLzrWaX5M+fm7nL5drYuXZ27hU795rpeQMVp9BtlGLXH2Blt1jZv/J7CeXlZSb/OfZDEdlHVNn3Udl/7y4iPqCkyyfVD0VlX/I6rfXoIjoMCur373ofislu451iPbuIbVXn3r+bfygo6yhZw8j+u9rrPZT7o7/+wR5UTSMrjq6giDU5jrmorCLbUoE/K9AeoSXjfgM7UKvD3R4AAA==",
    "agent_runs_powered.json": "...",
    "classical_runs.json": "...",
    "per_start_table.csv": "H4sIAFXa9WoC/81cXY8dN459318x79soiBSpjx8yz0bPuDNrrNMO2s5gMr9+ztFt21cSb98KFhsHQTo2TZdYEnl4SLHy+PLzwz8+PX58+Pz09P7hyyd5+Pzl0y/v/v748ePnyy+fXl4efvrw/Phx/Oqf+IOn3979/Pj0jn/69fcfnmX6Pf/86fn9w3gOf/VfL4/P7z/9/N9U+svLrx+fHh6ff3tID+oHfhh+SDpqlVx7yj2pWy2RqB4t1V6amqTkqvkhHVJzqqXrt5+BFpeIbZBrG+oh2rRm1eQm3TQStaPklIpr7trNc4ENswGl50Drtg16ZUOuh3Ys2KtZxZpqkagdXhp2pSgWrC037oPnZuX7Tw+0btuQr/fhyFK1S1fpUnr1Eony0WptWCr1VlwqTZg2QbPuSrctsCsLFOvVbtlqEi0i1SNRObThhKvhNVWaemjCrnXbBr/ehX4UsYS/0Jq7d9dQJEdJlmu27qVgl4VGLGfRI7XbVpTpLDzl1LWlDkdySz0S+dFKdccRK1xNksEIXcKiBVq3bajXp6GHCx6QEYNY1FKORPA2ODvOJeOAKsyjDctOWKB1bcP7p49fHt9CCTwx48VLgWM7HxuLkkOGncaazTXRlA0lAq17plyDRcYGuPURE25qWiIRrPOWxJXbXWqGP6ZjCVSLtO6ZAo381ZR0dByBpO5whqqlWiSSo6q2Vg2QKsZ/oDXZgZ8lUrtny4QdtRw41iK9JAJn0lCEcFAz/K6oWsWKdJY2b4tFavdsuUYR0QMvnvHXasqIuPX3ejQFLhn+6bn3zmyS1x0JtO7ZMKFIBW6pufTsPeF4PRQ5MCOXDljBEkg2JQT0QO2eLV+xJI3oEe2IE4SJ5UJYhqhkJLXivSIGkKjoJz2XbAqkrfDITEBzWdKbHqnBjIqfuWJPT1gyIcqRUrKGnOLVOjJEjkVWkX1HMPVajCnG2hTGLVAahvz6/OGnTy9vMo52aMFe9sxThXsFEj8Qzth+E8HLNs0eglqg9oYR1yjihyMewBAQHUjvqQYSOyrRHrhQurWqEpKeXesNE3RhXg0oXnsv/IX2SIRXJJmSxNyTmvUbGWbVesOIPBsB6xldXRM5U41EThQHJCF9ZayAFBwm20DtDTNsOg6EF/K7gHAmgJUFEoBSF4OPJsRNS+1ixIIbkdYbNkyI4YjJjASNGBBYLxqJ/CAiNpLLjmUkpD6b0hsmlCk8jRQFYAnCgv96jkQOQgZiagIPAbmRmIzvWm8Y8YoRPPqHaqAr7gxIt9LA9yAWA4HyVJHVFBj+YMAm0IiMlJlbdzBusvElvxZDdOD9YTx2JeNF6rIVN3iHfc+wKeMNsPngng3/LRLKAOzNwAurEsCs0BqbTyVSumvLAhmg9AL0xpPxmEiASgUMEbuHwEmFuLkcSw2U7lqh045klBYtATAd/0ky3nWXIeekVjPSS8/IPr2QdbgvxgRad63Jk8PCM4VpmukUxKVFItCsDExgCUV0gAvBlrKCeaB11xabdqZmVwQ8khKY/sUlNxGrGcRRr1lYl+APb7CxTe2uMTP9GLkJEegJBWluNRQh+QIo6An4aTqKOZ1LWo207toylTKlgZW6lOStoCBFXR6JsP+1APFRJ6DgA9OhLR7J765+TTkGApighAT/A6LXQILKqoLSNMAUK5tSNOI+gdarJX9/evny+OH5y29v0A4DvRIWo6g9rIHaBhIAHYoq0PaMh+f8imp99o4Sqd0xROY8g7MH5zZQS6QIhuEuKnjZ0izhX0sCyIo8Y1e6Y4e+ifMGmo7HpoYqvxVAmw3+2jrCoIIR4d17uCEVZABJGoYgOyDi7Z4dE4bUw7EYmC+jIjNhb5ICl8EBwQFR/YNb6I2Ow6p1xwybi20kyt4bTjUhvdC7dlE9+HDEDc7Cm42ycq8Xdq07hlxDB35kRGXtKAxAfJBjI1E/QJZZyoCXA/A7YSz50g4rgdodSybgQHpAjrVMPtcuWDALcFCwxhsKm1YRnHZjO1atO0bcoSOdfT8ceEPBnysyKCRd2JURQHXVZDnkp3DTBhQDlVES/Shs7zVC9PAMkg2qXxK9vQUS1HY49Q7+lyzBvpaj+inQOmXNBCWHsc1gyRh6UnKLRTgEZ13RM3+I3mJHi9Ype6aShpsO3AAggRmybRqLisCwTNRiM290UQN7Nq1T9uSpV4ToNcSBV0cGkV3AnGIIVDAPL+w1E+A8Ep9afAKUA67ugtIYlMzowZEI54UEArArOti0WZz6Nq1T9vjsuoCA1AAFxuZODyTsVID3KKAGwQbGcos1rlqnrClsJn9jazhRPhPRCKzMqpFIDpQOCbUNmTaq/FrCVlGgdsqgOvkuik2UFOzao4gG5MQiq7VXNnPgkvDLC/tey55AbRj0y+OX/3l3jq9g8Y4YdhRPhQWoSiRCyrMuyDUoiZRlVgh8u9YZaxbSAuaFApDFS/XRS9xF5agO4HFr4P3MnGFlumudsUbnawoUclhWGyDO6y6oCPSibIiA3fEuKuoV7EpnDLkGmM7Ky5SVcQFZ2X7vB8g+Qb6x5tNBn/b8uGudsWPquI4TLo5aDidRGKObxMcdSmWHCDm8txstnEXpjCVT4QNW7Vg28abD2N6MRAVcxRNpJqSgvDls7+1aZ6wpM5cCU0dZjIVZlppHogK6Rj7PpjDOYIDM5iy70hljrhEGfp8EFVwROAjwswUSP+DBqFeB9VgIBOFGu2/Vim25f63DezE+uLF5WTSU8GZFSQwQsfBtibP1pnXeJOF177e8IJk3BXhI9Za7BRLUw0yHJObsRpWBepbWsArUztukU+mI51XguDRw6WS7QHlNK2o4korKRdtIC2WOrhKpnbdoJjaJ1XNuAi4OalVKJLpcKWkVGJtJl2MeuiqdN2mmO44gzVqBf+xv5RqJkMK4F02N+ctGwRClz1XrvE0TGCG+GntQldkGZUSLRPATZIFqhDyUV15vutOqdt6q+WoZ3IBtSKtskrXBqDaRoliR0iSDixdQELtxSbZqnbepTnRM1bPwQokv2S8Xg6tID15KZEtjR0rc4NyVhkmfH9+///j07ulfv3z89OHL2yUWTr0LyGViD04CScEa7L0iuWWWLfXSq9rF95deWQ6iAcyzFRCmzjvtXdQPAZ7gjFCRwsF7DXsiu9Z9W3SxBU8YJ5D7aF1ukoZKAWGSOq/iy6Xtvef0Xeu+Jfn79VBQgSdeD+OsgWBgC2ps++LYC6/RAUBOGmbheA6yaGYjQnpHMs1fK4c3bVmITha4Y2cooFJtORLBDzKv4VEAgz74KHgv7rGI7y++TKYQFgpKOJLMwjmDQCQHX855SQEOxUu8G5Mpq9p9a2YYaS4FgCGJ/ct2KVhWUQfZlg5yzhs5gF7MQHet+7bMTd6MAgABom54gnkgqYebcAgFxKI2aS2m5KtSZMh9XoNsC76Amq2jyrcxmxKIAOYG2G3YfKDLaObtSTtQO2uTTLCGLAucJtMTDlN5JGJ6SqjIef+bRrcvzEWB2lmblsskUHIFqwWsgFjKGKrbZXqgbiSkw09hbo2z9q511qY8uRKQDT7M+1swN3jALsFhYuMK6tlBL9i6jy9xNrWzFtlEtrTBLTh7ZsCwJKEEr00fYb/AAXcWpcZA66xBM61BrZqyF1IFTmhqJNIDW4UD7G4ARbuk9L1pHqidtWkhNVIKNqXy7j+9+vIuwm61MZxJeMrj/tyWaiLSOmvS3NEpeLXsXVixufLmJhIxEzmHtpzXUXIj4FatYdHPT8+/PvxV3v3t8eXlw9PLAzsCY/7pcnXtnctdUmUeobzLFNDbAEq1Gl7Wclzx7VpcZFtfkOy/ry9Ih5zjVYCEpQvZDoSXRMTbBXoPcDff2oJVLbRBL6xBXu/vx7/4tfemKPGFg0ZCCpGX66xASaI9ZnpP4z3PbrEh/VYFNlXQWGxhPG+2a7Vgdbteffw7Cq/CZj5YKnZotAT37QvUoufTjosXnX07zqYho8E9wbyA1h7S0V2rB6uXy8Ca6tX7IUuRnVRDBBuSQvIb6L+paeQe9XUJuVqCjU3BvgvwJzMyX5vwq5R/6fWB+u4y8/417186CIcoEjSCFKSKLeRAIofC/8GP4MfFDE+/MRm6anGFdXG5mvI7szgco7OZjJPjTKP2McTU1jwRqXGNdfkxGZ0uE6np27+FBAomA6ocUURA23h4oMWnrAvkC57478ITBBKLd05dd/ZWx3z4DmmBGhdaTbgE3OQufJ/G69CSDIhRXyc9ljZzoBX5zyXgJP+uV0wHJ7sAVsMzeUnF5WSJuUBLgk0uD6/Fxe+zAEkIrwbIVH5nILeOedWSvltQL0e8+BHeAX8ZhW9GyVkvL9RW0I7UvnnSu59ePv376fkqR0+5crqoY5nQcnUQPTwokOi4A0fFSexjEWm3emyr2rdU/aY9coE9tQmYcuf8M8oRlnh13A4uuxwonVtQ58k/8C8OPSQrr22UXcIuWqsCV1arr82zoCW0qZ2z55WFD1fACRY8LlUbVrTByhBIqFXYxncxErVyIGNSUuiUbUBpMASYkS46jhK1DtN7+zorfNciW+b+UZ4qHCI3FqmBRFH/IsoKOxMo1eABl8I/kJ8zwKfCBKmAww+c6iyt10ACxMdrK8gDr/VhVw0Lk0DtnD1lSnaNbS626dkeu8yKspNbQVQrG0WdOy8dAF/T6ODly63PBpW8l244UjFeW+LYz+5Pvcp/iFjLLvwmxZixE4M4p066wyt8eImTi7ELJaVyUhF4VEMfzgAa5eQtAK+w+fSmPSsX+MGQcs0O/ghEuaIDfwZAuSIPfw48uaISPwROrqjGnwFNrnjHnwFMrkjIH4ol//v86W+fb/ERvAP4Vkali/XE6vjyJhClxHsJTs+4lF5uhM+qFa8/D3ZV0n7Q8oLdB3drkYij1kCm7mDQuY1Pd4KyOtCKDdDprhoVC8grSsXq7Jh4JMq8oFflpGjixbiERH/XitefLzgz/j7AWvi5o9sooDcRCyy35FqtwuOqhpNkgVZsgM0dbUXlJ+xEa+YnW7uEfareyH1R8CJWStTz25XixZcGHwfFK9ybU6S9BxIOhaO+6ZxLra6uN7Bh1YpXL8vUHG+ni+EdCmc2IxHgCs9twEFEY9Lxudo+wrFrxevPH6mNsRycMKczOhsJmyTzKwpEAqTjkKvf+Ax6U7te/wZxyPxEBKjDTI+EUCIRTCoEajxaZGxt+CVloBYaIMvFjmOTPXWgb+c0ayBCViYKc1+sp5T7jfVXrXD5OfSZ7qvlTLrjZoEEr4VEA5EaxztlvH1wE7xphavPX5SApAH0gdP8XnR89hmJBIgGQsLS3vJA3pGYV3G43tc4zxekByw5oL17HrPWqyDzg5ZSeudMfMt5TGjuOx2ojTp4Xfz/Euf8grz43UB/VQvfvUwY18qYx/KUMztVgYSff8B3EXrpMvoTDJkvOuG6dYow6JI88f/rgITZAskYlcmqXRPw9rVjHOTWVev74qebD4LtNCbI3Dvon5hEIuX/WMK98WvPVPWWPYvSSXNkvpRzMDCEWhbusF4u5TYZQht4zG+2OCfdxuepHkhPmjAPPGlKnTyrsmcE8hSJ4HIcpq4AN5CB2ixmH6vSSXtmUEDVIbzB4z3/+MY2kIgp+TvwuQEgPfpgZlc6ac1MDfiFMHioElc5Q7VLlLdaFQUT3h7s9DLHFvnLpnbSoGWyG3Qkg5Mh0fLi1iIR6hv4jfA2DgiHcH0dmgnkJ234EaThVDvi9bsz5Dz+fzcSB/5rCSRwTo6CATx6HwPJ3/LIJn/TgpBH/DBYuSYVPwZVQl7x40DlBtH4MZgSdiN+IKSExOSPRZSQnPxAQAlJy/8HnvwHSBtL8wdMAAA=",
}
for name, b64 in EMBED.items():
    path = f"{WORKDIR}/data/{name}"
    if not os.path.exists(path):
        open(path, "wb").write(gzip.decompress(base64.b64decode(b64)))
print("WORKDIR:", WORKDIR); print("NB10 data:", sorted(os.listdir(f"{WORKDIR}/data")))


In [ ]:
#@title 3. Shared code (fpes.py). Identical in all notebooks.
%%writefile fpes.py
"""fpes — shared code for the "Fruitful PES Explorer" notebooks.

HCN <-> HNC on a rigid two-coordinate grid (theta, r) around the C-N centre of mass,
several electronic-structure fidelities (xTB, MACE, PBE, B3LYP, Skala, CCSD(T)),
a grid "oracle" so that acquisition experiments are fast and reproducible,
baseline acquisition policies, and two LLM "Judge" designs (step agent, rule agent).

Energies are stored in Hartree; barriers are reported in kcal/mol.
"""
from __future__ import annotations
import os, json, time, math, dataclasses, textwrap, warnings
import numpy as np
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
try:
    from sklearn.exceptions import ConvergenceWarning
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
except Exception:
    pass

HARTREE_KCAL = 627.5094740631
EV_HARTREE = 1.0 / 27.211386245988
BOHR = 0.529177210903

# ----------------------------------------------------------------------------- geometry
M_C, M_N = 12.0, 14.003074
R_CN_DEFAULT = 1.16  # Angstrom, between HCN (1.153) and HNC (1.169)


def hcn_geometry(theta_deg: float, r: float, R_CN: float = R_CN_DEFAULT):
    """Return [(symbol, (x,y,z)), ...] in Angstrom.

    C and N lie on the z axis with their centre of mass at the origin.
    theta = 0 puts H on the C side (HCN), theta = 180 on the N side (HNC).
    r is the H distance from the C-N centre of mass.
    """
    zc = -R_CN * M_N / (M_C + M_N)
    zn = R_CN * M_C / (M_C + M_N)
    t = math.radians(theta_deg)
    h = (r * math.sin(t), 0.0, -r * math.cos(t))
    return [("C", (0.0, 0.0, zc)), ("N", (0.0, 0.0, zn)), ("H", h)]


def atoms_to_pyscf(atoms):
    return "; ".join(f"{s} {x:.6f} {y:.6f} {z:.6f}" for s, (x, y, z) in atoms)


def make_grid(n_theta: int, n_r: int, theta_range=(0.0, 180.0), r_range=(1.05, 2.30)):
    theta = np.linspace(*theta_range, n_theta)
    r = np.linspace(*r_range, n_r)
    return theta, r


# ----------------------------------------------------------------------------- hardware plan
def detect_gpu():
    try:
        import subprocess
        out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=10).stdout.strip()
        return out.split("\n")[0] if out else None
    except Exception:
        return None


def plan_for_hardware(gpu_name: str | None):
    """Grid density and basis per accelerator so that each notebook fits ~3 h."""
    g = (gpu_name or "").upper()
    if "A100" in g or "H100" in g:
        return dict(tier="A100", n_theta=37, n_r=25, basis="def2-tzvp", skala_every=1)
    if "L4" in g:
        return dict(tier="L4", n_theta=31, n_r=21, basis="def2-svp", skala_every=1)
    if "T4" in g:
        return dict(tier="T4", n_theta=25, n_r=17, basis="def2-svp", skala_every=1)
    return dict(tier="CPU", n_theta=13, n_r=9, basis="def2-svp", skala_every=2)


# ----------------------------------------------------------------------------- calculators
class Calc:
    """Uniform interface: calc(atoms) -> dict(E=hartree, converged=bool, niter=int, seconds=float)."""
    name = "base"

    def __call__(self, atoms):
        raise NotImplementedError


class XTB(Calc):
    name = "xtb"

    def __init__(self, method="GFN2-xTB"):
        from tblite.interface import Calculator  # noqa
        self._Calculator = Calculator
        self.method = method
        self._Z = {"H": 1, "C": 6, "N": 7, "O": 8}

    def __call__(self, atoms):
        t = time.time()
        numbers = np.array([self._Z[s] for s, _ in atoms])
        pos = np.array([c for _, c in atoms]) / BOHR
        calc = self._Calculator(self.method, numbers, pos)
        calc.set("verbosity", 0)
        try:
            res = calc.singlepoint()
            e = float(res.get("energy"))
            ok = True
        except Exception:
            e, ok = float("nan"), False
        return dict(E=e, converged=ok, niter=-1, seconds=time.time() - t)


class MACE(Calc):
    name = "mace"

    def __init__(self, model="small", device="cpu"):
        from mace.calculators import mace_off  # noqa
        from ase import Atoms  # noqa
        self._Atoms = Atoms
        self.calc = mace_off(model=model, device=device, default_dtype="float64")

    def __call__(self, atoms):
        t = time.time()
        a = self._Atoms([s for s, _ in atoms], positions=[c for _, c in atoms])
        a.calc = self.calc
        e = a.get_potential_energy() * EV_HARTREE
        return dict(E=float(e), converged=True, niter=-1, seconds=time.time() - t)


class DFT(Calc):
    """PySCF Kohn-Sham. xc in {'pbe','b3lyp',...}. gpu=True uses gpu4pyscf when importable."""

    def __init__(self, xc="pbe", basis="def2-svp", gpu=False, grids_level=3):
        from pyscf import gto, dft  # noqa
        self.gto, self.dft = gto, dft
        self.xc, self.basis, self.gpu, self.grids_level = xc, basis, gpu, grids_level
        self.name = xc
        self._dm0 = None
        if gpu:
            try:
                import gpu4pyscf  # noqa
                self._gpu_ok = True
            except Exception:
                self._gpu_ok = False
        else:
            self._gpu_ok = False

    def reset_guess(self):
        self._dm0 = None

    def __call__(self, atoms):
        t = time.time()
        mol = self.gto.M(atom=atoms_to_pyscf(atoms), basis=self.basis, unit="Angstrom", cart=False, verbose=0)
        mf = self.dft.RKS(mol)
        mf.xc = self.xc
        mf.grids.level = self.grids_level
        if self._gpu_ok:
            mf = mf.to_gpu()
        mf.conv_tol = 1e-9
        mf.max_cycle = 100
        try:
            e = mf.kernel(dm0=self._dm0)
            ok = bool(mf.converged)
            dm = mf.make_rdm1()
            self._dm0 = dm.get() if hasattr(dm, "get") else dm
            niter = int(getattr(mf, "cycles", -1) or -1)
        except Exception:
            e, ok, niter = float("nan"), False, -1
            self._dm0 = None
        return dict(E=float(e), converged=ok, niter=niter, seconds=time.time() - t)


class Skala(Calc):
    """Microsoft Skala learned XC through its PySCF bindings.

    CPU: from skala.pyscf import SkalaKS ; GPU: from skala.gpu4pyscf import SkalaKS.
    Weights download from Hugging Face on first use.
    """
    name = "skala"

    def __init__(self, basis="def2-svp", gpu=False, xc="skala-1.1", with_dftd3=False):
        from pyscf import gto  # noqa
        self.gto = gto
        if gpu:
            from skala.gpu4pyscf import SkalaKS  # noqa
        else:
            from skala.pyscf import SkalaKS  # noqa
        self.SkalaKS, self.basis, self.xc, self.with_dftd3 = SkalaKS, basis, xc, with_dftd3
        self._dm0 = None

    def reset_guess(self):
        self._dm0 = None

    def __call__(self, atoms):
        t = time.time()
        mol = self.gto.M(atom=atoms_to_pyscf(atoms), basis=self.basis, unit="Angstrom", cart=False, verbose=0)
        try:
            try:
                mf = self.SkalaKS(mol, xc=self.xc, with_dftd3=self.with_dftd3)
            except TypeError:  # older signature without with_dftd3
                mf = self.SkalaKS(mol, xc=self.xc)
            mf.conv_tol = 1e-9
            mf.max_cycle = 100
            e = mf.kernel(dm0=self._dm0)
            ok = bool(mf.converged)
            dm = mf.make_rdm1()
            self._dm0 = dm.get() if hasattr(dm, "get") else dm
            niter = int(getattr(mf, "cycles", -1) or -1)
        except Exception as ex:  # keep going; record the failure
            e, ok, niter = float("nan"), False, -1
            self._dm0 = None
            print("Skala failed:", repr(ex)[:200])
        return dict(E=float(e), converged=ok, niter=niter, seconds=time.time() - t)


class CCSDT(Calc):
    """CCSD(T) reference with PySCF (RHF-based). Closed-shell only."""
    name = "ccsd_t"

    def __init__(self, basis="def2-tzvp"):
        from pyscf import gto, scf, cc  # noqa
        self.gto, self.scf, self.cc, self.basis = gto, scf, cc, basis

    def __call__(self, atoms):
        t = time.time()
        mol = self.gto.M(atom=atoms_to_pyscf(atoms), basis=self.basis, unit="Angstrom", cart=False, verbose=0)
        try:
            mf = self.scf.RHF(mol).run(conv_tol=1e-10)
            mycc = self.cc.CCSD(mf).run()
            et = mycc.ccsd_t()
            e = mf.e_tot + mycc.e_corr + et
            ok = bool(mf.converged and mycc.converged)
        except Exception:
            e, ok = float("nan"), False
        return dict(E=float(e), converged=ok, niter=-1, seconds=time.time() - t)


# ----------------------------------------------------------------------------- surfaces with checkpoints
def compute_surface(calc: Calc, theta, r, path: str, every: int = 1, log_every: int = 10, max_minutes: float | None = None):
    """Evaluate calc on the (theta, r) grid; resume from an .npz checkpoint if present.

    `every` > 1 evaluates a coarser sub-grid (every k-th theta and r) and leaves the rest NaN.
    `max_minutes` stops gracefully (checkpoint kept) so a Colab session limit never loses work; rerun to resume.
    Returns dict with E (n_theta, n_r), converged, niter, seconds arrays.
    """
    nt, nr = len(theta), len(r)
    if os.path.exists(path):
        d = dict(np.load(path, allow_pickle=True))
        if d["E"].shape == (nt, nr):
            E, conv, nit, sec = d["E"], d["converged"], d["niter"], d["seconds"]
        else:
            E = np.full((nt, nr), np.nan); conv = np.zeros((nt, nr), bool); nit = -np.ones((nt, nr), int); sec = np.zeros((nt, nr))
    else:
        E = np.full((nt, nr), np.nan); conv = np.zeros((nt, nr), bool); nit = -np.ones((nt, nr), int); sec = np.zeros((nt, nr))
    done_mask = ~np.isnan(E)
    todo = [(i, j) for i in range(0, nt, every) for j in range(0, nr, every) if not done_mask[i, j]]
    print(f"[{calc.name}] grid {nt}x{nr}, every={every}: {len(todo)} points to compute, {done_mask.sum()} cached")
    t0 = time.time()
    last_i = None
    for k, (i, j) in enumerate(todo):
        if hasattr(calc, "reset_guess") and last_i != i:
            calc.reset_guess()  # new theta row: fresh SCF guess
        res = calc(hcn_geometry(theta[i], r[j]))
        E[i, j], conv[i, j], nit[i, j], sec[i, j] = res["E"], res["converged"], res["niter"], res["seconds"]
        last_i = i
        if max_minutes is not None and (time.time() - t0) / 60 > max_minutes:
            np.savez(path, E=E, converged=conv, niter=nit, seconds=sec, theta=theta, r=r)
            print(f"  time box of {max_minutes} min reached after {k+1} points; checkpoint saved, rerun this cell to resume")
            break
        if (k + 1) % log_every == 0 or k == len(todo) - 1:
            np.savez(path, E=E, converged=conv, niter=nit, seconds=sec, theta=theta, r=r)
            el = time.time() - t0
            print(f"  {k+1}/{len(todo)}  theta={theta[i]:.0f} r={r[j]:.2f}  E={E[i,j]:.6f}  "
                  f"conv={conv[i,j]}  {sec[i,j]:.1f}s/pt  elapsed {el/60:.1f} min  ETA {(el/(k+1))*(len(todo)-k-1)/60:.1f} min")
    np.savez(path, E=E, converged=conv, niter=nit, seconds=sec, theta=theta, r=r)
    return dict(E=E, converged=conv, niter=nit, seconds=sec, theta=theta, r=r)


def fill_nan_grid(E, theta, r):
    """Fill NaN entries (coarse Skala sub-grid or failed points) by bilinear interpolation."""
    from scipy.interpolate import griddata
    T, R = np.meshgrid(theta, r, indexing="ij")
    ok = ~np.isnan(E)
    if ok.all():
        return E.copy()
    pts = np.column_stack([T[ok] / 180.0, (R[ok] - r.min()) / (r.max() - r.min())])
    q = np.column_stack([T.ravel() / 180.0, (R.ravel() - r.min()) / (r.max() - r.min())])
    lin = griddata(pts, E[ok], q, method="linear")
    near = griddata(pts, E[ok], q, method="nearest")
    out = np.where(np.isnan(lin), near, lin).reshape(E.shape)
    return out


# ----------------------------------------------------------------------------- barrier on a grid
def basin_minima(E, theta):
    """Indices of the HCN (theta<45) and HNC (theta>135) minima."""
    hcn = np.where(theta < 45.0)[0]
    hnc = np.where(theta > 135.0)[0]
    sub = np.where(np.isnan(E), np.inf, E)
    i1 = np.unravel_index(np.argmin(sub[hcn]), sub[hcn].shape); a = (hcn[i1[0]], i1[1])
    i2 = np.unravel_index(np.argmin(sub[hnc]), sub[hnc].shape); b = (hnc[i2[0]], i2[1])
    return a, b


def bottleneck_barrier(E, theta):
    """Minimum-bottleneck path between the two basins on a 4-connected grid.

    Returns dict(barrier_kcal, saddle=(i,j), E_hcn, E_hnc, isomerization_kcal, threshold).
    """
    nt, nr = E.shape
    Ew = np.where(np.isnan(E), np.inf, E)
    a, b = basin_minima(E, theta)
    order = np.argsort(Ew, axis=None)
    parent = np.arange(nt * nr)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    active = np.zeros(nt * nr, bool)
    ia, ib = a[0] * nr + a[1], b[0] * nr + b[1]
    saddle = None
    for flat in order:
        if not np.isfinite(Ew.flat[flat]):
            break
        active[flat] = True
        i, j = divmod(int(flat), nr)
        for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            ni, nj = i + di, j + dj
            if 0 <= ni < nt and 0 <= nj < nr and active[ni * nr + nj]:
                ra, rb = find(flat), find(ni * nr + nj)
                if ra != rb:
                    parent[ra] = rb
        if find(ia) == find(ib):
            saddle = (i, j)
            break
    if saddle is None:
        return dict(barrier_kcal=float("nan"), saddle=None, E_hcn=E[a], E_hnc=E[b], isomerization_kcal=float("nan"))
    thr = E[saddle]
    return dict(barrier_kcal=float((thr - E[a]) * HARTREE_KCAL), saddle=saddle, hcn=a, hnc=b,
                E_hcn=float(E[a]), E_hnc=float(E[b]), isomerization_kcal=float((E[b] - E[a]) * HARTREE_KCAL),
                threshold=float(thr))


def corridor_mask(E, theta, margin_kcal=6.0):
    """Points within `margin` below the bottleneck threshold that connect the two basins
    (the reaction valley near the saddle). Used by path-aware acquisition rules."""
    info = bottleneck_barrier(E, theta)
    if info["saddle"] is None:
        return np.zeros_like(E, bool), info
    thr = info["threshold"]
    low = E <= thr + 1e-12
    # flood fill from the HCN minimum within the low set
    nt, nr = E.shape
    seen = np.zeros_like(low)
    stack = [info["hcn"]]
    while stack:
        i, j = stack.pop()
        if seen[i, j] or not low[i, j]:
            continue
        seen[i, j] = True
        for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            ni, nj = i + di, j + dj
            if 0 <= ni < nt and 0 <= nj < nr:
                stack.append((ni, nj))
    band = (E >= thr - margin_kcal / HARTREE_KCAL)
    return seen & band, info


# ----------------------------------------------------------------------------- oracle + surrogate
class GridOracle:
    """Expensive-method lookup on the dense reference grid. Counts calls."""

    def __init__(self, E_ref, theta, r, name="skala"):
        self.E, self.theta, self.r, self.name = E_ref, theta, r, name
        self.calls = 0
        self.log = []

    def nearest(self, th, rr):
        i = int(np.argmin(np.abs(self.theta - th))); j = int(np.argmin(np.abs(self.r - rr)))
        return i, j

    def __call__(self, i, j):
        self.calls += 1
        self.log.append((int(i), int(j), float(self.E[i, j])))
        return float(self.E[i, j])


def features(theta, r, idx):
    """Normalised (theta, r) features for GP regression."""
    th = theta[[i for i, _ in idx]] / 180.0
    rr = (r[[j for _, j in idx]] - r.min()) / (r.max() - r.min())
    return np.column_stack([th, rr])


def gp_predict(theta, r, idx, y, baseline=None, clip_kcal=150.0):
    """GP on sampled points -> (mean, std) over the full grid.

    If `baseline` (n_theta x n_r cheap surface, Hartree) is given, the GP models y - baseline
    (delta-learning) and the prediction adds baseline back.
    """
    from sklearn.gaussian_process import GaussianProcessRegressor
    from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C
    X = features(theta, r, idx)
    y = np.asarray(y, float)
    if baseline is not None:
        y = y - np.array([baseline[i, j] for i, j in idx])
    elif clip_kcal is not None:  # hard-wall points carry no information about the barrier; soften them
        y = np.minimum(y, y.min() + clip_kcal / HARTREE_KCAL)
    mu, sd = y.mean(), max(y.std(), 1e-6)
    yn = (y - mu) / sd
    kern = C(1.0, (1e-2, 1e2)) * RBF(length_scale=[0.3, 0.3], length_scale_bounds=(0.03, 3.0)) + WhiteKernel(1e-6, (1e-9, 1e-2))
    gp = GaussianProcessRegressor(kern, normalize_y=False, n_restarts_optimizer=2, random_state=0)
    gp.fit(X, yn)
    T, R = np.meshgrid(theta, r, indexing="ij")
    Xg = np.column_stack([T.ravel() / 180.0, (R.ravel() - r.min()) / (r.max() - r.min())])
    m, s = gp.predict(Xg, return_std=True)
    mean = (m * sd + mu).reshape(T.shape)
    std = (s * sd).reshape(T.shape)
    if baseline is not None:
        mean = mean + baseline
    return mean, std


# ----------------------------------------------------------------------------- acquisition baselines
def initial_design(theta, r, n_init, rng):
    """Always include the two basin guesses (HCN, HNC) plus random points."""
    nt, nr = len(theta), len(r)
    idx = [(0, int(np.argmin(np.abs(r - 1.69)))), (nt - 1, int(np.argmin(np.abs(r - 1.53))))]
    while len(idx) < n_init:
        c = (int(rng.integers(nt)), int(rng.integers(nr)))
        if c not in idx:
            idx.append(c)
    return idx


def choose_next(policy, theta, r, idx, mean, std, xtb=None, rng=None):
    nt, nr = len(theta), len(r)
    sampled = np.zeros((nt, nr), bool)
    for i, j in idx:
        sampled[i, j] = True
    if policy == "random":
        cand = np.argwhere(~sampled)
        return tuple(cand[rng.integers(len(cand))])
    if policy == "uniform":
        # farthest-point in feature space (space-filling)
        T, R = np.meshgrid(theta / 180.0, (r - r.min()) / (r.max() - r.min()), indexing="ij")
        pts = np.column_stack([T.ravel(), R.ravel()])
        S = features(theta, r, idx)
        d = np.min(((pts[:, None, :] - S[None, :, :]) ** 2).sum(-1), axis=1).reshape(nt, nr)
        d[sampled] = -1
        return tuple(np.unravel_index(np.argmax(d), d.shape))
    if policy == "uncertainty":
        a = std.copy(); a[sampled] = -1
        return tuple(np.unravel_index(np.argmax(a), a.shape))
    if policy in ("path_uncertainty", "saddle_exploit"):
        mask, info = corridor_mask(mean, theta)
        a = std * (1.0 + 4.0 * mask)
        if policy == "saddle_exploit" and info["saddle"] is not None and not sampled[info["saddle"]]:
            if rng.random() < 0.5:
                return info["saddle"]
        a[sampled] = -1
        return tuple(np.unravel_index(np.argmax(a), a.shape))
    raise ValueError(policy)


def run_policy(policy, oracle: GridOracle, theta, r, budget, seed=0, n_init=4, baseline=None, ref_barrier=None):
    """Sequential acquisition with a GP surrogate; returns per-call barrier estimate and error."""
    rng = np.random.default_rng(seed)
    oracle.calls = 0; oracle.log = []
    idx = initial_design(theta, r, n_init, rng)
    y = [oracle(i, j) for i, j in idx]
    hist = []
    while True:
        mean, std = gp_predict(theta, r, idx, y, baseline=baseline)
        for i, j in idx:  # trust sampled points exactly
            mean[i, j] = y[idx.index((i, j))]
        est = bottleneck_barrier(mean, theta)["barrier_kcal"]
        hist.append(dict(calls=len(idx), barrier=est, error=(est - ref_barrier) if ref_barrier is not None else None))
        if len(idx) >= budget:
            break
        nxt = choose_next(policy, theta, r, idx, mean, std, rng=rng)
        idx.append(tuple(int(v) for v in nxt)); y.append(oracle(*nxt))
    return hist, idx, y


# ----------------------------------------------------------------------------- LLM Judge
STEP_SYSTEM = """You are the Judge in a multi-fidelity exploration of a chemical potential energy surface.
Goal (stated by the user): {usecase}
The surface is HCN <-> HNC on a rigid grid: theta in degrees (0 = H on carbon, HCN; 180 = H on nitrogen, HNC),
r in Angstrom (H distance from the C-N centre of mass). A cheap xTB surface of the whole grid is available for free.
Each expensive call returns one Skala energy at a grid point and costs budget. You have {budget} expensive calls in total.
Use the tools. Prefer calls that reduce uncertainty about the quantity the goal asks for. Explain each choice in one sentence.
When the budget is exhausted, or when you judge the estimate converged, call finish."""

STEP_TOOLS = [
    dict(name="state_summary", description="Current surrogate summary: estimated barrier, predicted saddle, most uncertain points, corridor points, calls used.",
         input_schema=dict(type="object", properties={}, additionalProperties=False), strict=True),
    dict(name="xtb_cut", description="Free: a 1D cut of the cheap xTB surface. axis='theta' scans theta at fixed r; axis='r' scans r at fixed theta.",
         input_schema=dict(type="object", properties=dict(axis=dict(type="string", enum=["theta", "r"]), value=dict(type="number")),
                           required=["axis", "value"], additionalProperties=False), strict=True),
    dict(name="skala_point", description="Spend one expensive call: Skala energy at the grid point nearest (theta, r).",
         input_schema=dict(type="object", properties=dict(theta=dict(type="number"), r=dict(type="number"), why=dict(type="string")),
                           required=["theta", "r", "why"], additionalProperties=False), strict=True),
    dict(name="finish", description="Stop exploring and report the final barrier estimate.",
         input_schema=dict(type="object", properties=dict(reason=dict(type="string")), required=["reason"], additionalProperties=False), strict=True),
]

RULE_SYSTEM = """You design acquisition rules for multi-fidelity exploration of a chemical potential energy surface.
Goal (stated by the user): {usecase}
Surface: HCN <-> HNC, rigid grid over theta (deg, 0=HCN side, 180=HNC side) and r (Angstrom, H distance from the C-N centre of mass).
Write ONE Python function

    def score(theta, r, mean, std, sampled, xtb, corridor, info):
        ...
        return S  # ndarray shape (len(theta), len(r)); the unsampled argmax is the next expensive call

Arguments: theta (1D), r (1D), mean/std (2D surrogate prediction in Hartree over the grid), sampled (2D bool),
xtb (2D cheap surface, Hartree), corridor (2D bool: points in the predicted reaction valley near the bottleneck),
info (dict with keys hcn, hnc, saddle as (i,j) tuples, barrier_kcal). numpy is available as np. 1 Hartree = 627.5 kcal/mol.
Return only the code in a single ```python block. Be specific to the goal; do not just return std."""


class _MockLLM:
    """Deterministic stand-in used when no API key is present so the notebooks run end to end."""

    def step(self, state, calls_left):
        if calls_left <= 0:
            return dict(tool="finish", input=dict(reason="budget exhausted"))
        if state["saddle"] is not None and not state["saddle_sampled"]:
            th, rr = state["saddle_theta_r"]
            return dict(tool="skala_point", input=dict(theta=th, r=rr, why="confirm predicted saddle"))
        th, rr = state["top_uncertain"][0]
        return dict(tool="skala_point", input=dict(theta=th, r=rr, why="largest uncertainty in corridor"))

    def rule(self):
        return textwrap.dedent("""
        def score(theta, r, mean, std, sampled, xtb, corridor, info):
            S = std * (1.0 + 4.0 * corridor)
            if info.get('saddle') is not None and not sampled[info['saddle']]:
                S[info['saddle']] += 10 * std.max()
            return S
        """)


class JudgeRun:
    """Shared machinery for both agent designs: surrogate state, tool execution, logging."""

    def __init__(self, oracle, theta, r, xtb_grid, budget, ref_barrier=None, baseline=None, seed=0, n_init=4):
        self.oracle, self.theta, self.r, self.xtb, self.budget = oracle, theta, r, xtb_grid, budget
        self.ref, self.baseline = ref_barrier, baseline
        rng = np.random.default_rng(seed)
        oracle.calls = 0; oracle.log = []
        self.idx, self.y, self.failed = [], [], set()
        for (i, j) in initial_design(theta, r, n_init, rng):
            self._eval(i, j)
        tries = 0
        while len(self.idx) < n_init and tries < 6 * n_init:   # replace failed initial points (live mode only)
            tries += 1
            c = (int(rng.integers(len(theta))), int(rng.integers(len(r))))
            if c not in self.idx and c not in self.failed:
                self._eval(*c)
        self.hist, self.decisions = [], []
        self.tokens = dict(input=0, output=0)
        self.seed = seed
        self.notes, self.menu_log, self.policy_log, self.summary, self.model = [], [], [], None, None
        self.refresh()

    def _eval(self, i, j):
        e = self.oracle(i, j)
        if np.isfinite(e):
            self.idx.append((i, j)); self.y.append(e)
            return True
        self.failed.add((i, j))
        return False

    def refresh(self):
        self.mean, self.std = gp_predict(self.theta, self.r, self.idx, self.y, baseline=self.baseline)
        for k, (i, j) in enumerate(self.idx):
            self.mean[i, j] = self.y[k]
        self.corridor, self.info = corridor_mask(self.mean, self.theta)
        est = self.info["barrier_kcal"]
        self.hist.append(dict(calls=len(self.idx), barrier=est, error=(est - self.ref) if self.ref is not None else None))

    def sampled_mask(self):
        m = np.zeros_like(self.mean, bool)
        for i, j in list(self.idx) + list(self.failed):
            m[i, j] = True
        return m

    def state(self, k=6):
        s = self.std.copy(); s[self.sampled_mask()] = -1
        top = [tuple(np.unravel_index(f, s.shape)) for f in np.argsort(s, axis=None)[::-1][:k]]
        sad = self.info.get("saddle")
        return dict(
            calls_used=len(self.idx), calls_left=self.budget - len(self.idx),
            barrier_kcal=round(float(self.info["barrier_kcal"]), 2) if np.isfinite(self.info["barrier_kcal"]) else None,
            isomerization_kcal=round(float(self.info["isomerization_kcal"]), 2) if np.isfinite(self.info["isomerization_kcal"]) else None,
            saddle=sad, saddle_theta_r=(float(self.theta[sad[0]]), float(self.r[sad[1]])) if sad else None,
            saddle_sampled=bool(self.sampled_mask()[sad]) if sad else False,
            saddle_std_kcal=round(float(self.std[sad] * HARTREE_KCAL), 2) if sad else None,
            top_uncertain=[(float(self.theta[i]), float(self.r[j])) for i, j in top],
            top_uncertain_std_kcal=[round(float(self.std[i, j] * HARTREE_KCAL), 2) for i, j in top],
            corridor_points=int(self.corridor.sum()),
            sampled=[(float(self.theta[i]), float(self.r[j]), round(float((self.y[k] - min(self.y)) * HARTREE_KCAL), 1)) for k, (i, j) in enumerate(self.idx)],
        )

    def xtb_cut(self, axis, value):
        if axis == "theta":
            j = int(np.argmin(np.abs(self.r - value)))
            prof = (self.xtb[:, j] - np.nanmin(self.xtb)) * HARTREE_KCAL
            return "theta(deg): E_rel(kcal/mol) at r=%.2f\n" % self.r[j] + ", ".join(f"{t:.0f}:{e:.1f}" for t, e in zip(self.theta, prof))
        i = int(np.argmin(np.abs(self.theta - value)))
        prof = (self.xtb[i, :] - np.nanmin(self.xtb)) * HARTREE_KCAL
        return "r(A): E_rel(kcal/mol) at theta=%.0f\n" % self.theta[i] + ", ".join(f"{rr:.2f}:{e:.1f}" for rr, e in zip(self.r, prof))

    def spend(self, th, rr, why=""):
        i, j = self.oracle.nearest(th, rr)
        if (i, j) in self.idx:
            return f"({self.theta[i]:.0f}, {self.r[j]:.2f}) already sampled; no budget spent. E_rel={((self.y[self.idx.index((i,j))]-min(self.y))*HARTREE_KCAL):.2f} kcal/mol"
        if (i, j) in self.failed:
            return f"({self.theta[i]:.0f}, {self.r[j]:.2f}) failed earlier (SCF); no budget spent. Pick another point."
        if len(self.idx) >= self.budget:
            return "budget exhausted"
        if not self._eval(i, j):
            return f"Skala SCF failed at (theta={self.theta[i]:.0f}, r={self.r[j]:.2f}); the point is excluded and no budget is counted. Calls left {self.budget-len(self.idx)}."
        e = self.y[-1]
        self.decisions.append(dict(theta=float(self.theta[i]), r=float(self.r[j]), why=why))
        self.refresh()
        return f"Skala E_rel at (theta={self.theta[i]:.0f}, r={self.r[j]:.2f}) = {((e-min(self.y))*HARTREE_KCAL):.2f} kcal/mol. New barrier estimate {self.info['barrier_kcal']:.2f} kcal/mol. Calls left {self.budget-len(self.idx)}."


# ----------------------------------------------------------------------------- provider-neutral chat (Claude tool calling or Gemini function calling)
class Chat:
    """Tiny adapter: one conversation with function calling.

    send(text=None, tool_results=None) -> (assistant_text, [ {id, name, input} ... ])
    tool_results is a list of (call_id, name, result_string) answering the previous calls.
    """
    provider = "mock"

    def __init__(self, system: str, tools=None, model: str | None = None):
        self.system, self.tools, self.model = system, tools or [], model
        self.tokens = dict(input=0, output=0)

    def send(self, text=None, tool_results=None):
        raise NotImplementedError


class AnthropicChat(Chat):
    provider = "anthropic"
    DEFAULT = "claude-opus-5-5"

    def __init__(self, system, tools=None, model=None):
        super().__init__(system, tools, model or self.DEFAULT)
        import anthropic
        self.client = anthropic.Anthropic()
        self.messages = []

    def send(self, text=None, tool_results=None):
        if tool_results:
            self.messages.append({"role": "user", "content": [dict(type="tool_result", tool_use_id=cid, content=out) for cid, _, out in tool_results]})
        if text:
            self.messages.append({"role": "user", "content": text})
        kw = dict(model=self.model, max_tokens=4000, system=self.system, messages=self.messages)
        if self.tools:
            kw["tools"] = self.tools
        resp = self.client.messages.create(**kw)
        self.tokens["input"] += resp.usage.input_tokens; self.tokens["output"] += resp.usage.output_tokens
        self.messages.append({"role": "assistant", "content": resp.content})
        if resp.stop_reason == "refusal":
            return "[refused]", []
        txt = "".join(b.text for b in resp.content if b.type == "text")
        calls = [dict(id=b.id, name=b.name, input=dict(b.input)) for b in resp.content if b.type == "tool_use"]
        return txt, calls


# Gemini model ladder: the same ids as the C10H6 optimizer notebook. MODEL_FAST answers every Judge call;
# MODEL_PRO and MODEL_STRONG are used only if MODEL_FAST is unknown to the API (404) or keeps failing.
# Change them from a notebook with fpes.set_gemini_models(fast, pro, strong).
MODEL_FAST = "gemini-3.8-flash"
MODEL_PRO = "gemini-pro-latest"
MODEL_STRONG = "gemini-4-argon"
GEMINI_KEY_VARS = ("GEMINI_API_KEY", "GEMINI_API_KEY_2")   # a second key (optional) is used when the first is rate limited
GEMINI_UNAVAILABLE = set()   # model ids the API answered 404 for in this session; later chats skip them


def set_gemini_models(fast=None, pro=None, strong=None):
    """Set the Gemini ladder used by GeminiChat (blank or None keeps the current id)."""
    global MODEL_FAST, MODEL_PRO, MODEL_STRONG
    MODEL_FAST, MODEL_PRO, MODEL_STRONG = (fast or MODEL_FAST), (pro or MODEL_PRO), (strong or MODEL_STRONG)
    GeminiChat.DEFAULT = MODEL_FAST
    return [MODEL_FAST, MODEL_PRO, MODEL_STRONG]


def gemini_ladder(first=None):
    """Models to try, in order, without duplicates: the requested one first, then fast -> pro -> strong."""
    out = []
    for m in (first, MODEL_FAST, MODEL_PRO, MODEL_STRONG):
        if m and m not in out and m not in GEMINI_UNAVAILABLE:
            out.append(m)
    return out


def _gemini_error_kind(ex):
    """'model' = this model id is not available (go to the next model); 'transient' = rate limit / overload (wait, retry);
    'fatal' = bad request or bad key (raise)."""
    code = getattr(ex, "code", None) or getattr(ex, "status_code", None)
    msg = (str(getattr(ex, "status", "")) + " " + str(ex)).upper()
    if code == 404 or "NOT_FOUND" in msg or "IS NOT FOUND" in msg or "NOT SUPPORTED FOR GENERATECONTENT" in msg:
        return "model"
    if code in (429, 500, 502, 503, 504) or any(k in msg for k in ("RESOURCE_EXHAUSTED", "UNAVAILABLE", "OVERLOADED", "DEADLINE", "INTERNAL", "TIMEOUT", "TIMED OUT")):
        return "transient"
    return "fatal"


class GeminiChat(Chat):
    """Gemini function calling with a small safety net: key rotation, exponential back-off, model fallback.

    self.model is the model that answered the latest call (it changes only if the ladder had to move on);
    self.models_used counts calls per model so the cost column stays honest.
    """
    provider = "gemini"
    DEFAULT = MODEL_FAST
    MAX_ROUNDS_PER_MODEL = 3      # failed rounds (all keys) on one model before moving to the next model
    BACKOFF_BASE_S, BACKOFF_MAX_S = 5.0, 120.0

    def __init__(self, system, tools=None, model=None):
        super().__init__(system, tools, model or MODEL_FAST)
        from google import genai
        from google.genai import types
        self.types = types
        keys = [os.environ.get(v) for v in GEMINI_KEY_VARS] + [os.environ.get("GOOGLE_API_KEY")]
        keys = [k for i, k in enumerate(keys) if k and k not in keys[:i]]
        self.clients = [genai.Client(api_key=k) for k in keys] or [genai.Client()]
        self.client = self.clients[0]
        self.ladder = gemini_ladder(self.model)
        if not self.ladder:
            raise RuntimeError(f"Gemini: every model id was rejected earlier in this session ({sorted(GEMINI_UNAVAILABLE)}); fix the ids with fpes.set_gemini_models(...)")
        self.model = self.ladder[0]
        self.models_used, self.waited_s, self.retries = {}, 0.0, 0
        self.tokens_by_model = {}
        decls = []
        for t in self.tools:
            params = {k: v for k, v in t["input_schema"].items() if k != "additionalProperties"}
            decls.append(types.FunctionDeclaration(name=t["name"], description=t["description"], parameters=params))
        self.config = types.GenerateContentConfig(
            system_instruction=system,
            tools=[types.Tool(function_declarations=decls)] if decls else None,
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True) if decls else None,
        )
        self.contents = []
        self._n = 0

    def _generate(self):
        """One generate_content call with key rotation, back-off and model fallback. Raises only when every model failed."""
        import random
        last = None
        for model in list(self.ladder):
            for rnd in range(self.MAX_ROUNDS_PER_MODEL):
                for client in self.clients:
                    try:
                        resp = client.models.generate_content(model=model, contents=self.contents, config=self.config)
                        if model != self.model:
                            print(f"[gemini] answered by {model} (fallback from {self.model})")
                        self.model = model
                        self.models_used[model] = self.models_used.get(model, 0) + 1
                        return resp
                    except Exception as ex:   # noqa: BLE001  (the SDK raises several error classes)
                        last, kind = ex, _gemini_error_kind(ex)
                        if kind == "fatal":
                            raise
                        if kind == "model":
                            GEMINI_UNAVAILABLE.add(model)
                            print(f"[gemini] model {model} not available ({str(ex)[:80]}); trying the next model")
                            break
                        self.retries += 1
                else:   # every key hit a transient error: wait, then another round on the same model
                    wait = min(self.BACKOFF_MAX_S, self.BACKOFF_BASE_S * 2 ** rnd) * (0.8 + 0.4 * random.random())
                    print(f"[gemini] {model}: rate limited / busy ({str(last)[:60]}); waiting {wait:.0f} s")
                    time.sleep(wait); self.waited_s += wait
                    continue
                break   # model not available: leave the rounds loop, go to the next model
        raise RuntimeError(f"Gemini: no model in {self.ladder} answered; last error: {last!r}")

    def send(self, text=None, tool_results=None):
        T = self.types
        if tool_results:
            self.contents.append(T.Content(role="user", parts=[T.Part.from_function_response(name=name, response={"result": out}) for _, name, out in tool_results]))
        if text:
            self.contents.append(T.Content(role="user", parts=[T.Part.from_text(text=text)]))
        resp = self._generate()
        um = getattr(resp, "usage_metadata", None)
        if um is not None:
            ti, to = int(um.prompt_token_count or 0), int(um.candidates_token_count or 0) + int(getattr(um, "thoughts_token_count", 0) or 0)
            self.tokens["input"] += ti; self.tokens["output"] += to
            tm = self.tokens_by_model.setdefault(self.model, dict(input=0, output=0)); tm["input"] += ti; tm["output"] += to
        cand = resp.candidates[0] if resp.candidates else None
        if cand is None or cand.content is None:
            return "", []
        self.contents.append(cand.content)      # kept whole: Gemini 3 function calling needs its thought signatures back
        txt, calls = "", []
        for part in cand.content.parts or []:
            if getattr(part, "function_call", None):
                self._n += 1
                calls.append(dict(id=f"g{self._n}", name=part.function_call.name, input=dict(part.function_call.args or {})))
            elif getattr(part, "text", None) and not getattr(part, "thought", False):
                txt += part.text
        return txt, calls


def chat_factory(provider: str = "auto", model: str | None = None):
    """Return a callable (system, tools) -> Chat, or None for mock mode.

    provider: 'auto' picks Gemini if GEMINI_API_KEY is set, else Claude if ANTHROPIC_API_KEY is set, else mock.
    Colab secrets named GEMINI_API_KEY / GEMINI_API_KEY_2 / ANTHROPIC_API_KEY are read automatically.
    model: blank = MODEL_FAST for Gemini (with the fast -> pro -> strong fallback), claude-opus-5-5 for Claude.
    """
    for var in (*GEMINI_KEY_VARS, "ANTHROPIC_API_KEY"):
        if not os.environ.get(var):
            try:
                from google.colab import userdata  # noqa
                v = userdata.get(var)
                if v:
                    os.environ[var] = v
            except Exception:
                pass
    if provider == "auto":
        provider = "gemini" if os.environ.get("GEMINI_API_KEY") else ("anthropic" if os.environ.get("ANTHROPIC_API_KEY") else "mock")
    if provider == "mock":
        print("Judge in MOCK mode (scripted policy). Add GEMINI_API_KEY or ANTHROPIC_API_KEY as a Colab secret to use a real model.")
        return None
    if provider == "gemini":
        if not os.environ.get("GEMINI_API_KEY"):
            raise RuntimeError("GEMINI_API_KEY not set")
        n_keys = sum(bool(os.environ.get(v)) for v in GEMINI_KEY_VARS)
        print(f"Judge on Gemini: {gemini_ladder(model)} (first answers; the others are fallbacks) | API keys: {n_keys}")
        return lambda system, tools=None: GeminiChat(system, tools, model or None)
    if provider == "anthropic":
        if not os.environ.get("ANTHROPIC_API_KEY"):
            raise RuntimeError("ANTHROPIC_API_KEY not set")
        return lambda system, tools=None: AnthropicChat(system, tools, model or None)
    raise ValueError(provider)


# $ per million tokens (input, output); used only for the cost column.
# Models not listed here show cost NaN (unknown), never a fake $0. Add a price with fpes.PRICE["model-id"] = (in, out).
PRICE = {"claude-opus-5-5": (4.0, 20.0), "claude-sonnet-5-5": (2.0, 10.0), "claude-haiku-5-5": (0.10, 0.50),
         "gemini-2.5-flash": (0.30, 2.50), "gemini-2.5-pro": (1.25, 10.0)}
_PRICE_WARNED = set()


def cost_usd(model, tokens):
    if not model or str(model).startswith(("frozen", "mock")):
        return 0.0
    if model not in PRICE:
        if model not in _PRICE_WARNED:
            _PRICE_WARNED.add(model)
            print(f"[cost] no price for {model}: cost shows NaN. Set fpes.PRICE['{model}'] = (usd_per_M_in, usd_per_M_out) to fill it.")
        return float("nan")
    pi, po = PRICE[model]
    return (tokens["input"] * pi + tokens["output"] * po) / 1e6


def run_step_agent(judge: JudgeRun, usecase: str, chat_factory=None, max_turns=80, verbose=True, model=None, client=None):
    """Design A: the LLM chooses every expensive point itself, through function calls.

    chat_factory: from fpes.chat_factory(); None = scripted mock. (`client`/`model` kept for backward compatibility.)
    """
    if chat_factory is None and client is not None:
        chat_factory = lambda system, tools=None: AnthropicChat(system, tools, model)
    mock = chat_factory is None
    system = STEP_SYSTEM.format(usecase=usecase, budget=judge.budget)
    chat = None if mock else chat_factory(system, STEP_TOOLS)
    judge.model = "mock" if mock else chat.model
    pending_text, pending_results = "Begin. Call state_summary first.", None
    for turn in range(max_turns):
        if mock:
            st = judge.state()
            act = _MockLLM().step(st, st["calls_left"])
            calls = [dict(id=f"mock{turn}", name=act["tool"], input=act["input"])]
        else:
            txt, calls = chat.send(text=pending_text, tool_results=pending_results)
            judge.tokens = chat.tokens; judge.model = chat.model
            pending_text, pending_results = None, None
            if txt == "[refused]":
                print("model refused; stopping"); break
            if not calls:
                pending_text = "Continue: call a tool (skala_point, xtb_cut, state_summary) or finish."
                continue
        results, done = [], False
        for c in calls:
            name, inp = c["name"], c["input"]
            if name == "state_summary":
                out = json.dumps(judge.state())
            elif name == "xtb_cut":
                out = judge.xtb_cut(inp["axis"], float(inp["value"]))
            elif name == "skala_point":
                out = judge.spend(float(inp["theta"]), float(inp["r"]), str(inp.get("why", "")))
                if verbose:
                    print(f"  call {len(judge.idx):2d}: ({float(inp['theta']):.0f}, {float(inp['r']):.2f}) {str(inp.get('why',''))[:70]} -> {out[:60]}")
            elif name == "finish":
                out = "finished"; done = True
            else:
                out = f"unknown tool {name}"
            results.append((c["id"], name, out))
        pending_results = results
        if done or len(judge.idx) >= judge.budget:
            break
    return judge


def _compile_rule(code: str):
    code = code.strip()
    if "```" in code:
        code = code.split("```python")[-1].split("```")[0] if "```python" in code else code.split("```")[1]
    ns = {"np": np, "math": math}
    exec(compile(code, "<rule>", "exec"), ns)
    return ns["score"], code


def run_rule_agent(judge: JudgeRun, usecase: str, chat_factory=None, review_every=10, max_rewrites=3, verbose=True, model=None, client=None):
    """Design B: the LLM writes an acquisition rule once; code executes it; the LLM may rewrite after each review."""
    if chat_factory is None and client is not None:
        chat_factory = lambda system, tools=None: AnthropicChat(system, tools, model)
    mock = chat_factory is None
    chat = None if mock else chat_factory(RULE_SYSTEM.format(usecase=usecase), None)
    judge.model = "mock" if mock else chat.model
    rules = []

    def ask(prompt):
        if mock:
            return _MockLLM().rule()
        txt, _ = chat.send(text=prompt)
        judge.tokens = chat.tokens; judge.model = chat.model
        return txt

    score_fn, code = _compile_rule(ask("Write the initial acquisition rule.")); rules.append(code)
    rewrites = 0
    while len(judge.idx) < judge.budget:
        try:
            S = np.asarray(score_fn(judge.theta, judge.r, judge.mean, judge.std, judge.sampled_mask(), judge.xtb, judge.corridor, judge.info), float)
            S = np.where(np.isfinite(S), S, -np.inf)
        except Exception as ex:
            if verbose: print("rule failed, falling back to uncertainty:", repr(ex)[:100])
            S = judge.std.copy()
        S[judge.sampled_mask()] = -np.inf
        i, j = np.unravel_index(int(np.argmax(S)), S.shape)
        judge.spend(float(judge.theta[i]), float(judge.r[j]), why=f"rule v{len(rules)}")
        if verbose:
            print(f"  call {len(judge.idx):2d}: ({judge.theta[i]:.0f}, {judge.r[j]:.2f}) barrier est {judge.info['barrier_kcal']:.2f}")
        if len(judge.idx) % review_every == 0 and rewrites < max_rewrites and len(judge.idx) < judge.budget:
            review = (f"Review after {len(judge.idx)} calls. State: {json.dumps(judge.state(k=4))}. "
                      "Barrier estimates so far (kcal/mol): " + ", ".join(f"{h['barrier']:.1f}" for h in judge.hist[-review_every:] if np.isfinite(h['barrier'])) +
                      ". Keep the rule or return an improved one in a single ```python block.")
            try:
                score_fn, code = _compile_rule(ask(review)); rules.append(code); rewrites += 1
            except Exception as ex:
                if verbose: print("rewrite unusable, keeping previous rule:", repr(ex)[:100])
    judge.rules = rules
    return judge


# ----------------------------------------------------------------------------- Designs M (menu judge) and K (knob tuner)
# Design M: code builds a short, stratified menu of candidate grid points each turn; the LLM picks one by id.
# Design K: the LLM sets the weights of a fixed acquisition formula once per round; code runs the round.
# Both have a "frozen policy" twin (no LLM) that uses the same menus / the same formula, which is the ablation
# that measures the agent's contribution. Both also serve as the mock Judge when no API key is present.

_SHADES = " .:-=+*#%@"
KNOB_NAMES = ("w_unc", "w_path", "w_gap", "w_saddle", "w_spread", "w_wall")


def ascii_map(Z, theta, r, sampled, mark=None, lo=0.0, hi=80.0):
    """Small character map for the LLM: rows = r (top = largest), columns = theta. 'o' sampled, 'S' marked cell."""
    lines = []
    for j in range(len(r) - 1, -1, -1):
        row = ""
        for i in range(len(theta)):
            if mark is not None and (i, j) == tuple(mark):
                row += "S"
            elif sampled[i, j]:
                row += "o"
            else:
                v = (Z[i, j] - lo) / (hi - lo + 1e-12)
                row += _SHADES[min(9, max(0, int(v * 9.999)))] if np.isfinite(v) else "?"
        lines.append(f"r={r[j]:.2f} |{row}|")
    lines.append(" " * 8 + "θ=0" + " " * max(1, len(theta) - 7) + "θ=180")
    return "\n".join(lines)


def _norm01(a):
    a = np.where(np.isfinite(a), a, 0.0)
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo + 1e-12)


def judge_features(J):
    """Per-cell features: predicted E_rel and sigma (kcal/mol), |xTB - surrogate| gap (kcal/mol), grid distance to nearest sample."""
    sig = J.std * HARTREE_KCAL
    mu = (J.mean - J.mean.min()) * HARTREE_KCAL
    gap = np.abs((J.mean - J.mean.min()) - (J.xtb - np.nanmin(J.xtb))) * HARTREE_KCAL
    nt, nr = J.mean.shape
    T, R = np.meshgrid(np.arange(nt), np.arange(nr), indexing="ij")
    pts = np.column_stack([T.ravel(), R.ravel()]).astype(float)
    S = np.array(J.idx, float)
    dist = np.sqrt(((pts[:, None, :] - S[None, :, :]) ** 2).sum(-1)).min(1).reshape(nt, nr)
    return mu, sig, gap, dist


def build_menu(J, rng=None, box=None, want=None, k_max=8):
    """Stratified candidate menu: one unsampled cell per criterion, duplicates removed.

    box = (theta_min, theta_max, r_min, r_max) restricts the search (probe).
    want in {uncertainty, disagreement, low_energy, spread} ranks the whole box by one criterion instead (probe).
    """
    rng = rng if rng is not None else np.random.default_rng(getattr(J, "seed", 0))
    mu, sig, gap, dist = judge_features(J)
    allowed = ~J.sampled_mask()
    if box is not None:
        tmin, tmax, rmin, rmax = [float(x) for x in box]
        allowed &= (J.theta[:, None] >= tmin) & (J.theta[:, None] <= tmax) & (J.r[None, :] >= rmin) & (J.r[None, :] <= rmax)
    cands = []

    def add(tag, score, k=1, positive=False):
        s = np.where(allowed, score, -np.inf)
        for flat in np.argsort(s, axis=None)[::-1][:k]:
            i, j = (int(x) for x in np.unravel_index(int(flat), s.shape))
            if not np.isfinite(s[i, j]) or (positive and s[i, j] <= 0):
                break
            if all((i, j) != c[0] for c in cands):
                cands.append(((i, j), tag))

    if want is not None:
        crit = {"uncertainty": sig, "disagreement": gap, "low_energy": -mu, "spread": dist}.get(want, sig)
        add(str(want), crit, k=k_max)
    else:
        sad = J.info.get("saddle")
        if sad is not None and allowed[sad]:
            cands.append((tuple(int(x) for x in sad), "saddle guess"))
        add("max-σ on path", sig * J.corridor, positive=True)
        add("max-σ anywhere", sig)
        add("xTB≠GP on path", gap * J.corridor, positive=True)
        add("xTB≠GP anywhere", gap)
        add("far from samples", dist)
        basin_added = False
        for key in ("hcn", "hnc"):
            b = J.info.get(key)
            if b is not None and allowed[b] and all(tuple(b) != c[0] for c in cands):
                cands.append((tuple(int(x) for x in b), f"{key.upper()} basin min, unsampled")); basin_added = True; break
        if not basin_added:
            add("low E, unsampled", -mu)
        free = np.argwhere(allowed)
        if len(free):
            i, j = (int(x) for x in free[rng.integers(len(free))])
            if all((i, j) != c[0] for c in cands):
                cands.append(((i, j), "random"))
    rows = []
    for n, ((i, j), tag) in enumerate(cands[:k_max], 1):
        rows.append(dict(id=n, i=i, j=j, theta=float(J.theta[i]), r=float(J.r[j]), tag=tag,
                         E_pred=float(mu[i, j]), sigma=float(sig[i, j]), gap=float(gap[i, j]),
                         on_path=bool(J.corridor[i, j]), dist=float(dist[i, j])))
    return rows


def menu_text(rows):
    out = ["MENU (pick one by id)", " id    θ     r  why it is here              E_pred    σ   gap path dist"]
    for m in rows:
        out.append(f" [{m['id']}] {m['theta']:>3.0f}  {m['r']:.2f}  {m['tag']:<26} {m['E_pred']:>7.1f} {m['sigma']:>4.1f} {m['gap']:>5.1f}  {'Y' if m['on_path'] else 'n'}  {m['dist']:>4.1f}")
    return "\n".join(out)


def state_block(J):
    """What the LLM reads every turn: budget, estimate and trend, saddle and basins, two ASCII maps, its own notes."""
    mu, sig, gap, dist = judge_features(J)
    sampled = J.sampled_mask(); sad = J.info.get("saddle"); hcn, hnc = J.info.get("hcn"), J.info.get("hnc")
    est = J.info["barrier_kcal"]
    traj = ", ".join(f"{h['barrier']:.1f}" for h in J.hist[-6:] if np.isfinite(h["barrier"]))
    out = [f"BUDGET   used {len(J.idx)} / {J.budget}  (left {J.budget - len(J.idx)})",
           f"ESTIMATE barrier {est:.1f} kcal/mol" + (f", σ at saddle {sig[sad]:.1f}" if sad is not None else ""),
           f"         last estimates: {traj}"]
    if sad is not None:
        out.append(f"SADDLE   θ={J.theta[sad[0]]:.0f}°, r={J.r[sad[1]]:.2f} Å  ({'sampled' if sampled[sad] else 'NOT yet sampled at Skala level'})")
    if hcn is not None and hnc is not None:
        out.append(f"BASINS   HCN θ={J.theta[hcn[0]]:.0f} r={J.r[hcn[1]]:.2f} ({'sampled' if sampled[hcn] else 'unsampled'})")
        out.append(f"         HNC θ={J.theta[hnc[0]]:.0f} r={J.r[hnc[1]]:.2f} ({'sampled' if sampled[hnc] else 'unsampled'}); ΔE(HNC−HCN) {J.info['isomerization_kcal']:.1f}")
    if J.decisions:
        d = J.decisions[-1]
        out.append(f"LAST     ({d['theta']:.0f}, {d['r']:.2f}) because: {str(d['why'])[:120]}")
    out += ["", "MAP predicted E_rel, kcal/mol  (' '=0 … '@'≥80)", "    o = sampled, S = saddle guess",
            ascii_map(mu, J.theta, J.r, sampled, sad, 0, 80),
            "", "MAP uncertainty σ, kcal/mol  (' '=0 … '@'≥10)",
            ascii_map(sig, J.theta, J.r, sampled, sad, 0, 10),
            "", "NOTES (yours): " + ("; ".join(J.notes[-6:]) if J.notes else "none yet")]
    return "\n".join(out)


def pick_from_menu(J, menu, cid, why):
    """Spend one expensive call on menu item `cid`; log which kind of candidate was taken."""
    try:
        cid = int(cid)
    except Exception:
        return f"id must be an integer 1..{len(menu)}"
    m = next((x for x in menu if x["id"] == cid), None)
    if m is None:
        return f"no candidate with id {cid}; ids are 1..{len(menu)}"
    before = J.info["barrier_kcal"]
    out = J.spend(m["theta"], m["r"], str(why))
    J.menu_log.append(dict(call=len(J.idx), id=cid, tag=m["tag"], theta=m["theta"], r=m["r"], why=str(why)[:200],
                           offered=[x["tag"] for x in menu]))
    sad = J.info.get("saddle")
    extra = f" Barrier was {before:.1f}."
    if sad is not None:
        extra += f" Saddle guess now θ={J.theta[sad[0]]:.0f}, r={J.r[sad[1]]:.2f}; σ there {J.std[sad] * HARTREE_KCAL:.1f}."
    return out + extra


MENU_ORDER = """
DEFAULT DECISION ORDER (the GOAL overrides it; take the first that applies):
 1. Saddle guess unsampled and sigma there above 0.5: pick it.
 2. An on-path candidate has sigma above 1 or gap above 3: pick the one that most threatens the
    estimate (largest sigma + gap on the path).
 3. A basin minimum is unsampled: pick it; the barrier is measured from it.
 4. Estimate moved less than 0.5 over the last 5 picks and nothing above applies: finish.
 5. Otherwise the most uncertain candidate with E_pred below 80.
"""

MENU_SYSTEM = """You are the Judge of a multi-fidelity exploration of a chemical potential energy surface.

GOAL (from the user, verbatim): {usecase}
METRIC: the HCN -> HNC bottleneck barrier (the highest energy on the lowest connecting path
across the grid), within 1 kcal/mol of the converged answer, using as few expensive calls as
possible. Both count: the error and the number of calls.

THE SURFACE: HCN <-> HNC on a rigid grid. theta (degrees): 0 = H on the carbon side (HCN),
180 = H on the nitrogen side (HNC). r (Angstrom): distance of H from the C-N centre of mass.
Expect two basins (theta near 0-15 and near 165-180), one saddle around theta 70-100 at small r,
and steep walls (E_rel above 80 kcal/mol) at small r inside the basins and at large r everywhere.
The cheap xTB surface is roughly right about shape and often wrong about heights near the saddle.

FREE: the state block, the maps, the menu, probe, xtb_cut, note.  COSTS: pick, one Skala energy.
You have {budget} picks.

EACH TURN you get a state block (budget, estimate and its trend, saddle guess, two ASCII maps:
predicted E_rel and uncertainty sigma; 'o' = sampled, 'S' = saddle guess) and a MENU of
candidates with: E_pred, sigma, gap = |E_rel from xTB - E_rel from the surrogate|,
path = Y if the point lies in the reaction valley near the bottleneck, dist = grid steps to the
nearest sampled point.
{order}
RULES: exactly one function call per turn. Never pick E_pred above 80 unless the GOAL is about
walls. Never pick a point marked already sampled. Start 'why' with what the last result changed,
then the reason for this pick; 30 words at most. Call note() when you form or drop a hypothesis;
notes are shown back to you. A good estimate with budget left beats a spent budget."""

MENU_TOOLS = [
    dict(name="pick", description="Spend one expensive call on a menu candidate. The only function that costs budget.",
         input_schema=dict(type="object", properties=dict(id=dict(type="integer", description="menu id"), why=dict(type="string", description="what the last result changed, then the reason; 30 words max")),
                           required=["id", "why"], additionalProperties=False), strict=True),
    dict(name="probe", description="Free: rebuild the menu inside a box of the grid, ranked by one criterion.",
         input_schema=dict(type="object", properties=dict(theta_min=dict(type="number"), theta_max=dict(type="number"), r_min=dict(type="number"), r_max=dict(type="number"),
                                                          want=dict(type="string", enum=["uncertainty", "disagreement", "low_energy", "spread"])),
                           required=["theta_min", "theta_max", "r_min", "r_max", "want"], additionalProperties=False), strict=True),
    STEP_TOOLS[1],  # xtb_cut
    dict(name="note", description="Free: store a short hypothesis or observation; it is shown back to you in every later state block.",
         input_schema=dict(type="object", properties=dict(text=dict(type="string")), required=["text"], additionalProperties=False), strict=True),
    dict(name="finish", description="Stop exploring and report the final barrier estimate.",
         input_schema=dict(type="object", properties=dict(barrier_kcal=dict(type="number"), confidence=dict(type="string", enum=["high", "medium", "low"]), summary=dict(type="string")),
                           required=["barrier_kcal", "confidence", "summary"], additionalProperties=False), strict=True),
]

KNOB_SYSTEM = """You set the sampling policy of a multi-fidelity exploration of a chemical potential energy surface.

GOAL (from the user, verbatim): {usecase}
METRIC: the HCN -> HNC bottleneck barrier (the highest energy on the lowest connecting path
across the grid), within 1 kcal/mol of the converged answer, using as few expensive calls as
possible. Both count: the error and the number of calls.

THE SURFACE: HCN <-> HNC on a rigid grid. theta (degrees): 0 = H on the carbon side (HCN),
180 = H on the nitrogen side (HNC). r (Angstrom): distance of H from the C-N centre of mass.
Expect two basins (theta near 0-15 and near 165-180), one saddle around theta 70-100 at small r,
and steep walls (E_rel above 80 kcal/mol) at small r inside the basins and at large r everywhere.
The cheap xTB surface is roughly right about shape and often wrong about heights near the saddle.

FREE: the review block and xtb_cut.  COSTS: set_policy runs round_size Skala calls at once.
You have {budget} calls in total. Code picks each point by maximising
  score = w_unc*sigma + w_path*(sigma on the path) + w_gap*|xTB - GP| + w_saddle*[saddle guess]
        + w_spread*(distance to sampled points) - w_wall*[E_pred above 80]
with every term scaled 0..1 over the grid. You choose the six weights (0 to 2) and round_size (4-10).

DEFAULT SCHEDULE (the GOAL overrides it): early rounds locate (w_saddle, w_path high, w_spread
moderate, large rounds); middle rounds hunt disagreement (w_gap up); late rounds confirm (w_path
only, small rounds). Finish when the barrier trajectory is flat within 0.5 over two rounds.

FORMAT: one function call per turn. In 'why', name the number in the review that made you change
each weight you changed. Keeping the same weights is allowed; say why."""

KNOB_TOOLS = [
    dict(name="set_policy", description="Set the acquisition weights and run one round of expensive calls. Returns a review of the round.",
         input_schema=dict(type="object", properties=dict(w_unc=dict(type="number"), w_path=dict(type="number"), w_gap=dict(type="number"), w_saddle=dict(type="number"),
                                                          w_spread=dict(type="number"), w_wall=dict(type="number"), round_size=dict(type="integer"), why=dict(type="string")),
                           required=list(KNOB_NAMES) + ["round_size", "why"], additionalProperties=False), strict=True),
    STEP_TOOLS[1],  # xtb_cut
    MENU_TOOLS[-1],  # finish
]


def acquisition(J, w):
    """Weighted acquisition over the grid (design K); sampled cells are -inf."""
    mu, sig, gap, dist = judge_features(J)
    sad = np.zeros_like(sig)
    if J.info.get("saddle") is not None:
        sad[J.info["saddle"]] = 1.0
    A = (w.get("w_unc", 0.0) * _norm01(sig) + w.get("w_path", 0.0) * _norm01(sig * J.corridor) + w.get("w_gap", 0.0) * _norm01(gap)
         + w.get("w_saddle", 0.0) * sad + w.get("w_spread", 0.0) * _norm01(dist) - w.get("w_wall", 0.0) * (mu > 80))
    A[J.sampled_mask()] = -np.inf
    return A


def _sigma_path_max(J):
    sig = J.std * HARTREE_KCAL
    return float((sig * J.corridor).max()) if J.corridor.any() else float(sig.max())


def run_round(J, w, n, why=""):
    """Run one round of design K with weights w; returns the review block for the LLM."""
    w = {k: float(min(2.0, max(0.0, float(w.get(k, 0.0))))) for k in KNOB_NAMES}
    n = int(min(12, max(1, int(n))))
    before = _sigma_path_max(J)
    rows = []
    label = " ".join(f"{k}={v:g}" for k, v in w.items())
    for _ in range(n):
        if len(J.idx) >= J.budget:
            break
        A = acquisition(J, w)
        if not np.isfinite(A).any():
            break
        i, j = np.unravel_index(int(np.argmax(A)), A.shape)
        J.spend(float(J.theta[i]), float(J.r[j]), why=f"round {len(J.policy_log) + 1}: {label}")
        rows.append((float(J.theta[i]), float(J.r[j]), float((J.y[-1] - min(J.y)) * HARTREE_KCAL), float(J.info["barrier_kcal"])))
    after = _sigma_path_max(J)
    J.policy_log.append(dict(round=len(J.policy_log) + 1, weights=w, round_size=n, why=str(why)[:300], calls=len(J.idx),
                             barrier=float(J.info["barrier_kcal"]), sigma_path_before=before, sigma_path_after=after))
    return review_block(J, w, rows, before, after)


def review_block(J, w, rows, before, after):
    sampled = J.sampled_mask(); sad = J.info.get("saddle"); sig = J.std * HARTREE_KCAL
    out = ["ROUND DONE with weights", "  " + " ".join(f"{k}={v:g}" for k, v in w.items()),
           f"BUDGET used {len(J.idx)} / {J.budget}",
           "POINTS THIS ROUND  (θ, r) -> E_rel ; barrier estimate after"]
    out += [f"  ({th:>3.0f}, {rr:.2f}) -> {e:>6.1f} ; {b:.1f}" for th, rr, e, b in rows]
    out.append("BARRIER trajectory (all rounds):")
    out.append("  " + ", ".join(f"{h['barrier']:.1f}" for h in J.hist[-14:] if np.isfinite(h["barrier"])))
    out.append(f"σ on path: max {before:.1f} -> {after:.1f} kcal/mol")
    if sad is not None:
        out.append(f"saddle guess θ={J.theta[sad[0]]:.0f} r={J.r[sad[1]]:.2f} ({'sampled' if sampled[sad] else 'unsampled'}), σ there {sig[sad]:.1f}")
    out.append("MAP σ (o=sampled, S=saddle guess)")
    out.append(ascii_map(sig, J.theta, J.r, sampled, sad, 0, 10))
    out.append(f"Calls left {J.budget - len(J.idx)}. Reply with set_policy(...) for the next round, or finish(...).")
    return "\n".join(out)


def _flat(J, k=5, tol=0.5):
    b = [h["barrier"] for h in J.hist[-(k + 1):] if np.isfinite(h["barrier"])]
    return len(b) >= k + 1 and (max(b) - min(b)) < tol


def frozen_menu_choice(J, menu, rule="order"):
    """Chooser without an LLM. rule='order' follows MENU_ORDER literally (mock Judge and frozen twin);
    rule='max_sigma' always takes the most uncertain candidate below the wall."""
    def act(m, why):
        return dict(name="pick", input=dict(id=m["id"], why=why))
    below = [m for m in menu if m["E_pred"] < 80] or menu
    if rule == "max_sigma":
        return act(max(below, key=lambda m: m["sigma"]), "frozen: most uncertain candidate")
    sad = next((m for m in menu if m["tag"] == "saddle guess"), None)
    if sad is not None and sad["sigma"] > 0.5:
        return act(sad, "rule 1: saddle guess unsampled")
    path = [m for m in menu if m["on_path"] and (m["sigma"] > 1 or m["gap"] > 3)]
    if path:
        return act(max(path, key=lambda m: m["sigma"] + m["gap"]), "rule 2: on-path candidate threatens the estimate")
    basin = next((m for m in menu if "basin" in m["tag"]), None)
    if basin is not None:
        return act(basin, "rule 3: basin minimum unsampled")
    if _flat(J):
        return dict(name="finish", input=dict(barrier_kcal=float(J.info["barrier_kcal"]), confidence="medium", summary="rule 4: estimate flat over the last 5 picks"))
    return act(max(below, key=lambda m: m["sigma"]), "rule 5: most uncertain candidate below the wall")


def frozen_knob_schedule(J, round_no):
    """Weights without an LLM: the default schedule of KNOB_SYSTEM (mock Judge and frozen twin)."""
    if round_no == 1:
        return dict(w_unc=0.2, w_path=1.0, w_gap=0.3, w_saddle=1.0, w_spread=0.3, w_wall=1.0, round_size=8, why="frozen schedule: locate")
    if round_no == 2:
        return dict(w_unc=0.1, w_path=1.5, w_gap=0.6, w_saddle=0.5, w_spread=0.0, w_wall=1.0, round_size=6, why="frozen schedule: hunt disagreement")
    return dict(w_unc=0.0, w_path=1.0, w_gap=0.2, w_saddle=0.0, w_spread=0.0, w_wall=1.0, round_size=6, why="frozen schedule: confirm")


def run_menu_agent(J, usecase, chat_factory=None, scaffold=True, frozen=None, max_free=3, verbose=True):
    """Design M: the LLM picks one tagged candidate per turn from a code-built menu.

    chat_factory None, or frozen in {'order', 'max_sigma'}: no LLM, the frozen chooser decides (mock mode / frozen twin).
    scaffold False removes the default decision order from the prompt (ablation).
    Each pick is a fresh conversation; memory is the notes and the state block, so cost stays flat.
    """
    rng = np.random.default_rng(getattr(J, "seed", 0))
    system = MENU_SYSTEM.format(usecase=usecase, budget=J.budget, order=MENU_ORDER if scaffold else "\n")
    use_llm = chat_factory is not None and frozen is None
    J.model = None if use_llm else f"frozen-{frozen or 'order'}"
    done = False
    while len(J.idx) < J.budget and not done:
        menu = build_menu(J, rng)
        if not menu:
            break
        chat = None
        if use_llm:
            chat = chat_factory(system, MENU_TOOLS); J.model = chat.model
            txt, calls = chat.send(text=state_block(J) + "\n\n" + menu_text(menu))
        else:
            a = frozen_menu_choice(J, menu, frozen or "order")
            calls = [dict(id="frozen", name=a["name"], input=a["input"])]
        free = 0
        for _guard in range(12):
            if not calls:
                if chat is None:
                    break
                txt, calls = chat.send(text="Call exactly one function: pick, probe, xtb_cut, note or finish.")
                continue
            c = calls[0]; name, inp = c["name"], c["input"]
            if name == "pick":
                out = pick_from_menu(J, menu, inp.get("id", 0), inp.get("why", ""))
                if "no candidate" in out or "must be an integer" in out:
                    if chat is None:
                        break
                    txt, calls = chat.send(tool_results=[(c["id"], name, out)]); continue
                if verbose:
                    print(f"  call {len(J.idx):2d}: {out[:70]} | {str(inp.get('why', ''))[:60]}")
                break
            if name == "finish":
                J.summary = dict(inp); done = True
                if verbose:
                    print(f"  finish after {len(J.idx)} calls: {str(inp.get('summary', ''))[:80]}")
                break
            if name == "probe":
                try:
                    menu = build_menu(J, rng, box=(inp["theta_min"], inp["theta_max"], inp["r_min"], inp["r_max"]), want=inp.get("want"))
                    out = menu_text(menu) if menu else "no unsampled points in that box"
                except Exception as ex:
                    out = f"probe failed: {ex!r}"
            elif name == "xtb_cut":
                out = J.xtb_cut(inp.get("axis", "theta"), float(inp.get("value", 0)))
            elif name == "note":
                J.notes.append(str(inp.get("text", ""))[:200]); out = "noted"
            else:
                out = f"unknown function {name}"
            free += 1
            if free >= max_free:
                out += "\nFree calls for this turn are used up: pick or finish now."
            if chat is None:
                break
            txt, calls = chat.send(tool_results=[(c["id"], name, out)])
        if chat is not None:
            J.tokens["input"] += chat.tokens["input"]; J.tokens["output"] += chat.tokens["output"]; J.model = chat.model
    return J


def run_knob_agent(J, usecase, chat_factory=None, fixed_weights=None, max_rounds=8, verbose=True):
    """Design K: the LLM sets six acquisition weights per round; code runs the round and returns a review.

    chat_factory None: the frozen default schedule decides (mock mode / frozen twin).
    fixed_weights: the same weights every round (a notebook-2 baseline expressed as a policy).
    """
    use_llm = chat_factory is not None and fixed_weights is None
    system = KNOB_SYSTEM.format(usecase=usecase, budget=J.budget)
    chat = chat_factory(system, KNOB_TOOLS) if use_llm else None
    J.model = chat.model if use_llm else ("frozen-fixed" if fixed_weights else "frozen-schedule")
    pending_text = "Begin. The landscape before any policy:\n\n" + state_block(J) + "\n\nCall set_policy for round 1."
    pending_results = None
    rounds = 0
    for _turn in range(max_rounds * 3):
        if len(J.idx) >= J.budget or rounds >= max_rounds:
            break
        if use_llm:
            txt, calls = chat.send(text=pending_text, tool_results=pending_results)
            pending_text, pending_results = None, None
            J.tokens = chat.tokens; J.model = chat.model
            if not calls:
                pending_text = "Call set_policy or finish."; continue
            c = calls[0]; name, inp = c["name"], c["input"]
        else:
            inp = dict(fixed_weights, round_size=int(fixed_weights.get("round_size", 6)), why="fixed weights") if fixed_weights else frozen_knob_schedule(J, rounds + 1)
            name, c = "set_policy", dict(id="frozen")
        if name == "finish":
            J.summary = dict(inp)
            if verbose:
                print(f"  finish after {len(J.idx)} calls: {str(inp.get('summary', ''))[:80]}")
            break
        if name == "xtb_cut":
            pending_results = [(c["id"], name, J.xtb_cut(inp.get("axis", "theta"), float(inp.get("value", 0))))]; continue
        if name != "set_policy":
            pending_results = [(c["id"], name, "unknown function; call set_policy or finish")]; continue
        review = run_round(J, inp, inp.get("round_size", 6), inp.get("why", ""))
        rounds += 1
        if verbose:
            p = J.policy_log[-1]
            print(f"  round {p['round']}: {' '.join(f'{k}={v:g}' for k, v in p['weights'].items())} n={p['round_size']} -> barrier {p['barrier']:.2f}, σ_path {p['sigma_path_after']:.1f} | {p['why'][:60]}")
        pending_results = [(c["id"], "set_policy", review)]
        if not use_llm and len(J.policy_log) >= 3 and abs(J.policy_log[-1]["barrier"] - J.policy_log[-2]["barrier"]) < 0.5 and abs(J.policy_log[-2]["barrier"] - J.policy_log[-3]["barrier"]) < 0.5:
            J.summary = dict(barrier_kcal=float(J.info["barrier_kcal"]), confidence="medium", summary="frozen schedule: flat over two rounds")
            break
    return J


def anthropic_client_or_none():
    """Backward-compatible helper: returns an Anthropic client or None (mock)."""
    f = chat_factory("anthropic" if os.environ.get("ANTHROPIC_API_KEY") else "mock")
    if f is None:
        return None
    import anthropic
    return anthropic.Anthropic()


# ----------------------------------------------------------------------------- plotting
def plot_surface(ax, E, theta, r, idx=None, title="", levels=None, cmap="viridis", vmax_kcal=120):
    Er = (E - np.nanmin(E)) * HARTREE_KCAL
    Er = np.clip(Er, 0, vmax_kcal)
    T, R = np.meshgrid(theta, r, indexing="ij")
    cs = ax.contourf(T, R, Er, levels=levels or np.linspace(0, vmax_kcal, 25), cmap=cmap)
    ax.contour(T, R, Er, levels=[10, 20, 30, 40, 50, 60], colors="w", linewidths=0.5, alpha=0.6)
    if idx is not None:
        ax.scatter([theta[i] for i, _ in idx], [r[j] for _, j in idx], s=18, c="#F0A04B", edgecolor="k", linewidth=0.4, zorder=5)
    ax.set_xlabel("theta (deg)   0 = HCN, 180 = HNC"); ax.set_ylabel("r (Angstrom)"); ax.set_title(title)
    return cs


def summarize_curves(runs, budget):
    """runs: list of hist lists (one per seed). Returns calls, median |error|, q25, q75 arrays."""
    calls = np.arange(min(len(h) for h in runs))
    err = np.array([[abs(h[k]["error"]) if h[k]["error"] is not None and np.isfinite(h[k]["error"]) else np.nan for k in calls] for h in runs])
    return np.array([h[k]["calls"] for k in calls for h in runs[:1]]), np.nanmedian(err, 0), np.nanpercentile(err, 25, 0), np.nanpercentile(err, 75, 0)


# ----------------------------------------------------------------------------- minimax path (for NB4)
def bottleneck_path(E, theta):
    """Minimax (lowest-maximum) path between the two basin minima on the grid, as a list of (i, j)."""
    import heapq
    nt, nr = E.shape
    Ew = np.where(np.isnan(E), np.inf, E)
    a, b = basin_minima(E, theta)
    best = np.full((nt, nr), np.inf); prev = {}
    best[a] = Ew[a]
    pq = [(Ew[a], a)]
    while pq:
        c, (i, j) = heapq.heappop(pq)
        if c > best[i, j]:
            continue
        if (i, j) == b:
            break
        for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            ni, nj = i + di, j + dj
            if 0 <= ni < nt and 0 <= nj < nr:
                nc = max(c, Ew[ni, nj])
                if nc < best[ni, nj]:
                    best[ni, nj] = nc; prev[(ni, nj)] = (i, j); heapq.heappush(pq, (nc, (ni, nj)))
    path = [b]
    while path[-1] != a:
        path.append(prev[path[-1]])
    return path[::-1]


class LiveOracle(GridOracle):
    """Same interface as GridOracle but calls a real calculator at the grid geometry (live GPU mode).

    Every point is computed at most once and cached (optionally on disk), so several Judge runs can share it:
    `fresh_calls` / `fresh_seconds` count only real calculator calls. A point whose SCF fails or does not
    converge is retried once from a fresh guess; if it still fails it is marked failed and returns NaN, which
    JudgeRun excludes from the surrogate and from later menus.
    """

    def __init__(self, calc: Calc, theta, r, name="skala-live", cache_path=None, accept_unconverged=False, retries=1):
        shape = (len(theta), len(r))
        super().__init__(np.full(shape, np.nan), theta, r, name)
        self.calc, self.cache_path, self.accept_unconverged, self.retries = calc, cache_path, accept_unconverged, retries
        self.conv = np.zeros(shape, bool); self.sec = np.zeros(shape); self.failed = np.zeros(shape, bool)
        self.fresh_calls, self.fresh_seconds = 0, 0.0
        if cache_path and os.path.exists(cache_path):
            d = dict(np.load(cache_path, allow_pickle=True))
            if d["E"].shape == shape:
                self.E, self.conv, self.sec, self.failed = d["E"], d["conv"], d["sec"], d["failed"]
                print(f"[{name}] cache: {int((~np.isnan(self.E)).sum())} points loaded, {int(self.failed.sum())} marked failed")

    def save(self):
        if self.cache_path:
            np.savez(self.cache_path, E=self.E, conv=self.conv, sec=self.sec, failed=self.failed, theta=self.theta, r=self.r)

    def compute(self, i, j):
        """Real calculator call for cell (i, j) unless cached; returns the energy (NaN if failed)."""
        if np.isnan(self.E[i, j]) and not self.failed[i, j]:
            ok = False
            for _attempt in range(1 + self.retries):
                if hasattr(self.calc, "reset_guess"):
                    self.calc.reset_guess()
                res = self.calc(hcn_geometry(self.theta[i], self.r[j]))
                self.fresh_calls += 1; self.fresh_seconds += res["seconds"]
                if np.isfinite(res["E"]) and (res["converged"] or self.accept_unconverged):
                    self.E[i, j], self.conv[i, j], self.sec[i, j] = res["E"], res["converged"], res["seconds"]
                    ok = True
                    break
            if not ok:
                self.failed[i, j] = True
                print(f"  [{self.name}] SCF failed at theta={self.theta[i]:.0f} r={self.r[j]:.2f}; point excluded")
            self.save()
        return float(self.E[i, j])

    def __call__(self, i, j):
        self.compute(i, j)
        return super().__call__(i, j)


# ----------------------------------------------------------------------------- other small systems (NB4)
def nh3_geometry(umbrella_deg: float, r_NH: float = 1.012):
    """NH3 with the three H at angle `umbrella_deg` below the plane (0 = planar transition state, ~22 = equilibrium)."""
    t = math.radians(umbrella_deg)
    rho, z = r_NH * math.cos(t), -r_NH * math.sin(t)
    atoms = [("N", (0.0, 0.0, 0.0))]
    for k in range(3):
        p = 2 * math.pi * k / 3
        atoms.append(("H", (rho * math.cos(p), rho * math.sin(p), z)))
    return atoms


def ch4_geometry(stretch: float, r_CH: float = 1.087):
    """Methane with one C-H bond stretched to `stretch` Angstrom; the other three fixed at tetrahedral positions."""
    v = 1 / math.sqrt(3)
    dirs = [(v, v, v), (-v, -v, v), (-v, v, -v), (v, -v, -v)]
    atoms = [("C", (0.0, 0.0, 0.0))]
    for k, d in enumerate(dirs):
        L = stretch if k == 0 else r_CH
        atoms.append(("H", (d[0] * L, d[1] * L, d[2] * L)))
    return atoms


def scan_1d(geoms, calcs: dict, path: str):
    """Evaluate each calculator on a list of geometries; checkpoint to JSON; return {name: [E,...]}."""
    out = json.load(open(path)) if os.path.exists(path) else {}
    for name, calc in calcs.items():
        rec = out.get(name, {"E": [], "converged": [], "seconds": []})
        if hasattr(calc, "reset_guess"):
            calc.reset_guess()
        for k in range(len(rec["E"]), len(geoms)):
            res = calc(geoms[k])
            rec["E"].append(res["E"]); rec["converged"].append(res["converged"]); rec["seconds"].append(res["seconds"])
            out[name] = rec
            json.dump(out, open(path, "w"))
            print(f"  [{name}] point {k+1}/{len(geoms)}  E={res['E']:.6f} conv={res['converged']} {res['seconds']:.1f}s")
    return out


In [ ]:
#@title 3b. Molecule layer: re-label the shared code for vinyl alcohol -> acetaldehyde (text and constants only; asserted)
# fpes.py above is the shared file, identical in all notebooks. This cell rewrites ONLY labels, prompt text, the initial-design
# anchors, the map scale and the wall cut-off, and checks that every rewrite hits exactly the expected number of places.
WALL = 120   # kcal/mol; HCN used 80 for a rigid barrier of ~49, CH3NC 150 for ~92. This grid has a barrier of ~76 (xTB), so 120 keeps the same ratio.
SURFACE_OLD = """THE SURFACE: HCN <-> HNC on a rigid grid. theta (degrees): 0 = H on the carbon side (HCN),
180 = H on the nitrogen side (HNC). r (Angstrom): distance of H from the C-N centre of mass.
Expect two basins (theta near 0-15 and near 165-180), one saddle around theta 70-100 at small r,
and steep walls (E_rel above 80 kcal/mol) at small r inside the basins and at large r everywhere."""
SURFACE_NEW = f"""THE SURFACE: vinyl alcohol CH2=CH-OH <-> acetaldehyde CH3-CH=O, a 1,3 hydrogen shift on a rigid recipe grid. The three heavy atoms C1(H2)-C2(H)-O lie in a plane; the migrating hydrogen is placed by two numbers measured from the midpoint M of C1 and O.
theta (degrees): 0 = hydrogen on the oxygen side (enol, vinyl alcohol), 180 = on the C1 side (keto, acetaldehyde); the bridging position across the C1...O line is near theta 97.
r (Angstrom): distance of the hydrogen from M. The rest of the molecule follows by a fixed recipe (bond lengths morph from enol to keto values as the hydrogen passes the bridge; the CH2 turns and pyramidalises into CH3 as the hydrogen approaches C1).
Expect two basins (the enol well near theta 30 at r about 1.4, the keto well near theta 150 at r about 1.4, the keto well lower), one saddle near theta 90-100 at r about 1.0 (the hydrogen bridging C1 and O),
and no steep walls: the energy rises smoothly towards small r (up to about 110 kcal/mol at r 0.6 in the middle of the theta range) and the saddle itself is high (about 76 kcal/mol above the enol well). Energies above {WALL} kcal/mol would be unusual."""
src_ = open("fpes.py").read()
def rep(old, new, n):
    global src_
    assert src_.count(old) == n, f"patch '{old[:50]}': expected {n} hits, found {src_.count(old)}"
    src_ = src_.replace(old, new)
rep(SURFACE_OLD, SURFACE_NEW, 2)                                    # menu + knob prompts
rep("the HCN -> HNC bottleneck barrier", "the enol -> keto bottleneck barrier, vinyl alcohol to acetaldehyde (measured from the theta near 30 basin, which is the HIGHER of the two wells)", 2)
rep("The surface is HCN <-> HNC on a rigid grid: theta in degrees (0 = H on carbon, HCN; 180 = H on nitrogen, HNC),\nr in Angstrom (H distance from the C-N centre of mass).",
    "The surface is vinyl alcohol <-> acetaldehyde (1,3 hydrogen shift) on a rigid recipe grid: theta in degrees (0 = H on oxygen, enol; 180 = H on carbon C1, keto),\nr in Angstrom (distance of the migrating H from the midpoint of C1 and O).", 1)
rep("Surface: HCN <-> HNC, rigid grid over theta (deg, 0=HCN side, 180=HNC side) and r (Angstrom, H distance from the C-N centre of mass).",
    "Surface: vinyl alcohol <-> acetaldehyde, rigid recipe grid over theta (deg, 0=enol side, 180=keto side) and r (Angstrom, distance of the migrating H from the C1-O midpoint).", 1)
rep('f"BASINS   HCN θ=', 'f"BASINS   enol θ=', 1)
rep('f"         HNC θ=', 'f"         keto θ=', 1)
rep("ΔE(HNC−HCN)", "ΔE(keto−enol)", 1)
rep('f"{key.upper()} basin min, unsampled"', '"%s basin min, unsampled" % {"hcn": "enol", "hnc": "keto"}[key]', 1)
rep('"theta (deg)   0 = HCN, 180 = HNC"', '"theta (deg)   0 = enol, 180 = keto"', 1)
rep("np.abs(r - 1.69)", "np.abs(r - 1.40)", 1); rep("np.abs(r - 1.53)", "np.abs(r - 1.40)", 1)    # initial-design basin anchors: both wells sit near r = 1.4 A
# wall cut-off 80 -> WALL (everywhere it is a threshold or a label), plot range 120 -> 1.5*WALL
rep("hi=80.0", f"hi={float(WALL)}", 1)
rep("'@'≥80)", f"'@'≥{WALL})", 1)
rep("sampled, sad, 0, 80)", f"sampled, sad, 0, {WALL})", 1)
rep("E_pred below 80", f"E_pred below {WALL}", 1)
rep("Never pick E_pred above 80", f"Never pick E_pred above {WALL}", 1)
rep("[E_pred above 80]", f"[E_pred above {WALL}]", 1)
rep("(mu > 80)", f"(mu > {WALL})", 1)
rep('m["E_pred"] < 80', f'm["E_pred"] < {WALL}', 1)
rep("vmax_kcal=120", f"vmax_kcal={int(1.5*WALL)}", 1)
open("fpes.py", "w").write(src_)
print("fpes.py re-labelled for vinyl alcohol -> acetaldehyde; wall cut-off", WALL, "kcal/mol")


In [ ]:
#@title 3c. Hardware plan and the vinyl alcohol / acetaldehyde geometry (migrating H at (theta, r), rest of the molecule by a fixed recipe)
import importlib, fpes, math; importlib.reload(fpes)
GPU = fpes.detect_gpu(); PLAN = fpes.plan_for_hardware(GPU)
print("GPU:", GPU); print("plan:", PLAN)

def _smooth(x):
    x = min(max(x, 0.0), 1.0); return x * x * (3 - 2 * x)

def frame(s, enol=(1.33, 1.37, 127.0), keto=(1.50, 1.21, 124.0)):
    d12 = enol[0] + s * (keto[0] - enol[0]); d2O = enol[1] + s * (keto[1] - enol[1]); al = enol[2] + s * (keto[2] - enol[2])
    C2 = np.zeros(2); C1 = np.array([d12, 0.0]); a = math.radians(al)
    O = d2O * np.array([math.cos(a), math.sin(a)])
    return C2, C1, O

def enol_keto_geometry(theta_grid, r, L_CH2=(1.085, 1.09), HCH=(117.0, 108.0), R_C2H=1.10):
    """Vinyl alcohol CH2=CH-OH -> acetaldehyde CH3-CH=O, 1,3-H shift. Heavy atoms C1(H2), C2(H), O lie in the xy plane.
    The migrating H sits at polar position (psi, r) about M = midpoint of C1...O: psi 0 = toward O (enol side), 180 = toward C1 (keto side),
    90 = across the C1...O line, away from C2 (the bridging position). The grid angle theta (0-180) maps to psi = 20 + 130*theta/180, so the enol well is near theta 32 and the keto well near theta 152. The rest of the molecule follows theta by a fixed recipe:
    bond lengths and the C-C-O angle morph from enol to keto values (smooth step between psi 60 and 120), and the two CH2 hydrogens turn and pyramidalise
    as the migrating H approaches C1 (weight w from the H...C1 distance: 0 beyond 2.0 A, 1 inside 1.25 A)."""
    theta_deg = 20.0 + 130.0 * theta_grid / 180.0          # polar angle psi of the migrating H: the grid's theta 0..180 covers psi 20..150 degrees
    s = _smooth((theta_deg - 60.0) / 60.0)
    C2, C1, O = frame(s)
    M = 0.5 * (C1 + O)
    ux = (O - C1) / np.linalg.norm(O - C1); uy = (M - C2) / np.linalg.norm(M - C2)
    t = math.radians(theta_deg)
    H = M + r * (math.cos(t) * ux + math.sin(t) * uy)
    # C2-H: in plane, pointing away from both neighbours
    v = -((C1 - C2) / np.linalg.norm(C1 - C2) + (O - C2) / np.linalg.norm(O - C2)); v /= np.linalg.norm(v)
    H2 = C2 + R_C2H * v
    # spectator hydrogens on C1
    dHC = float(np.linalg.norm(H - C1)); w = _smooth((2.0 - dHC) / (2.0 - 1.25))
    L = L_CH2[0] + w * (L_CH2[1] - L_CH2[0]); beta = math.radians(HCH[0] + w * (HCH[1] - HCH[0]))
    u2 = (C2 - C1) / np.linalg.norm(C2 - C1); um = (H - C1) / dHC
    b_sp3 = -(u2 + um); b_sp3 /= np.linalg.norm(b_sp3)
    b_sp2 = -u2
    b = (1 - w) * b_sp2 + w * b_sp3; b /= np.linalg.norm(b)
    ez = np.array([0.0, 0.0, 1.0]); b3 = np.array([b[0], b[1], 0.0]); ep = np.cross(ez, b3)      # in-plane perpendicular to b
    rho = w * math.pi / 2                                                                         # CH2 turns from in-plane (0) to perpendicular (90 deg)
    dirs = []
    for sg in (1.0, -1.0):
        d = math.cos(beta / 2) * b3 + sg * math.sin(beta / 2) * (math.cos(rho) * ep + math.sin(rho) * ez)
        dirs.append(np.array([C1[0], C1[1], 0.0]) + L * d)
    # hard core: where the migrating H would sit closer than 0.95 A to a heavy atom (0.80 A to another H) it is pushed straight out to that distance.
    # This touches only the corners of the grid (theta near 0 or 180 at small r), which are walls anyway; it stops the grid from asking for overlapping atoms.
    Hp = np.array([H[0], H[1], 0.0])
    others = [(np.array([C1[0], C1[1], 0.0]), 0.95), (np.array([C2[0], C2[1], 0.0]), 0.95), (np.array([O[0], O[1], 0.0]), 0.95),
              (np.array([H2[0], H2[1], 0.0]), 0.80), (dirs[0], 0.80), (dirs[1], 0.80)]
    for _ in range(6):
        for P_, dmin in others:
            v_ = Hp - P_; n_ = float(np.linalg.norm(v_))
            if n_ < dmin: Hp = P_ + (v_ / n_ if n_ > 1e-9 else np.array([ux[0], ux[1], 0.0])) * dmin
    p3 = lambda p: (float(p[0]), float(p[1]), 0.0)
    atoms = [("C", p3(C1)), ("C", p3(C2)), ("O", p3(O)), ("H", tuple(float(x) for x in Hp)), ("H", p3(H2)),
             ("H", tuple(float(x) for x in dirs[0])), ("H", tuple(float(x) for x in dirs[1]))]
    return atoms

def polar_of(point, theta_hint_s):
    C2, C1, O = frame(theta_hint_s); M = 0.5 * (C1 + O)
    ux = (O - C1) / np.linalg.norm(O - C1); uy = (M - C2) / np.linalg.norm(M - C2)
    d = np.asarray(point) - M; return math.degrees(math.atan2(d @ uy, d @ ux)), float(np.linalg.norm(d))


fpes.hcn_geometry = enol_keto_geometry     # compute_surface, LiveOracle and the smoke test all look this name up in the fpes module
# sanity: bond lengths at the two wells, and the closest pair of atoms anywhere on the grid (no overlaps)
def _d(a, b): return float(np.linalg.norm(np.array(a[1]) - np.array(b[1])))
g = enol_keto_geometry(32, 1.4)
print(f"enol well (theta 32, r 1.4): C=C {_d(g[0], g[1]):.2f}, C-O {_d(g[1], g[2]):.2f}, O-H {_d(g[2], g[3]):.2f} A (expect about 1.33 / 1.37 / 0.97)")
g = enol_keto_geometry(152, 1.4)
print(f"keto well (theta 152, r 1.4): C-C {_d(g[0], g[1]):.2f}, C=O {_d(g[1], g[2]):.2f}, C1-H(migrated) {_d(g[0], g[3]):.2f} A (expect about 1.50 / 1.21 / 1.09)")
_mind = min(min(_d(g_[i], g_[j]) for i in range(7) for j in range(i)) for g_ in (enol_keto_geometry(t_, q_) for t_ in np.linspace(0, 180, 19) for q_ in np.linspace(0.6, 1.8, 13)))
print(f"closest atom pair anywhere on the 19 x 13 grid: {_mind:.2f} A")


In [ ]:
#@title 4. Settings: what to run, time boxes, which start to replay. Defaults are the 2-hour run.
SKALA_BASIS = ""         #@param {type:"string"}
# blank = by hardware: A100 def2-tzvp, L4/T4 def2-svp. NB10 used def2-svp on an L4; the stored grid is def2-svp, so keep it for a fair comparison.
RUN_DENSITY = True       #@param {type:"boolean"}
RUN_RELAXED = True       #@param {type:"boolean"}
RUN_MICRO   = True       #@param {type:"boolean"}
G1_MAX_MIN  = 70         #@param {type:"integer"}
# total time box of the microscope scan (step 3); a rerun of that cell resumes
G1_THETA_STEP = 5.0      #@param {type:"number"}
G1_R_STEP     = 0.05     #@param {type:"number"}
N_DENS = 9               #@param {type:"integer"}
N_RELAX_STEP = 0.1       #@param {type:"number"}
SEED_SHOW = 0            #@param {type:"integer"}
# start used for the replay and panel A. Start 0 is the typical one in NB10: the agent stopped itself at 12 calls (median 12) while the best classical rule ended with error 1.5 (median 1.7).
SEEDS, BUDGET = 8, 24
STOP_TOL, STOP_WINDOW, STOP_MIN_CALLS = 0.5, 5, 10
VALLEY_KCAL = 3.0
N_THETA, N_R = 19, 13
BASIS = SKALA_BASIS or PLAN["basis"]
theta, r = fpes.make_grid(N_THETA, N_R, r_range=(0.6, 1.8))
if not GPU: print("No GPU detected: steps 1-3 need one (or set SKALA_STAND_IN in cell 8 for a dry run).")
print(f"basis {BASIS}; time box for the microscope {G1_MAX_MIN} min; replay start {SEED_SHOW}")


In [ ]:
#@title 5. Load the NB10 results, the free xTB scout, and the shared helper functions
import pandas as pd, textwrap, shutil, glob, math
import matplotlib; import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
D = f"{WORKDIR}/data"
_d = dict(np.load(f"{D}/ref_skala_def2-svp_19x13.npz", allow_pickle=True))
assert _d["E"].shape == (N_THETA, N_R) and np.allclose(_d["theta"], theta) and np.allclose(_d["r"], r), "stored NB10 grid does not match this notebook's grid"
E_REF = _d["E"].copy(); SEC_NB10 = float(np.nanmean(_d["seconds"]))
agent_runs = json.load(open(f"{D}/agent_runs_powered.json")); classical = json.load(open(f"{D}/classical_runs.json"))
TAB10 = pd.read_csv(f"{D}/per_start_table.csv")
res_x = fpes.compute_surface(fpes.XTB(), theta, r, f"{WORKDIR}/ref_xtb_{N_THETA}x{N_R}.npz", log_every=1000)
E_xtb = fpes.fill_nan_grid(res_x["E"], theta, r)
bs = fpes.bottleneck_barrier(E_REF, theta); REF_BARRIER = bs["barrier_kcal"]; SAD = bs["saddle"]; EN_MIN, KE_MIN = bs["hcn"], bs["hnc"]
bx = fpes.bottleneck_barrier(E_xtb, theta)
print(f"NB10 Skala grid loaded: barrier {REF_BARRIER:.2f} kcal/mol, saddle theta {theta[SAD[0]]:.0f} r {r[SAD[1]]:.2f}; Skala {SEC_NB10:.1f} s/point in NB10")
print(f"xTB barrier {bx['barrier_kcal']:.2f}; stored runs: {sum(len(v) for v in agent_runs.values())} agent, {sum(len(v) for v in classical.values())} classical")
KCAL = fpes.HARTREE_KCAL
# ---- colours (same as the NB10 figures) ----
COL = {"menu": "#D97A12", "menu_frozen": "#E8B77A", "knobs": "#6B4FD8", "knobs_frozen": "#B7A8EE", "classical": "#555555", "control": "#1F8A5B"}
CMAP = "viridis"
# ---- shared helper functions ----
def calls_to_tol(hist, tol=1.0):
    errs = [abs(h["error"]) if h["error"] is not None and np.isfinite(h["error"]) else np.inf for h in hist]
    for q in range(len(errs)):
        if all(e <= tol for e in errs[q:]): return hist[q]["calls"]
    return None
def stop_index(hist, tol=STOP_TOL, window=STOP_WINDOW, min_calls=STOP_MIN_CALLS):
    b = [h["barrier"] for h in hist]
    for q in range(len(hist)):
        if hist[q]["calls"] < min_calls or q < window: continue
        w = b[q - window:q + 1]
        if np.all(np.isfinite(w)) and (max(w) - min(w)) < tol: return q
    return len(hist) - 1
def run_mean(idx_k, delta=True):
    """GP surface (kcal/mol above the Skala minimum) rebuilt from the first picks, measured points kept exactly."""
    ids = [tuple(int(v) for v in p) for p in idx_k]; y = [E_REF[i, j] for i, j in ids]
    mean, std = fpes.gp_predict(theta, r, ids, y, baseline=E_xtb if delta else None)
    for (i, j), v in zip(ids, y): mean[i, j] = v
    return mean, std
def draw_mol(ax, atoms, title="", size=1.0):
    """Ball-and-stick, viewed from above the heavy-atom plane. Hydrogens off the plane are drawn smaller."""
    P = np.array([c for _, c in atoms]); S = [s for s, _ in atoms]
    colr = {"C": "#6b6b6b", "O": "#d63b3b", "H": "#f2f2f2"}; rad = {"C": 0.30, "O": 0.28, "H": 0.17}
    n = len(atoms)
    for i in range(n):
        for j in range(i):
            d = np.linalg.norm(P[i] - P[j]); k = {S[i], S[j]}
            lim = 1.7 if k <= {"C", "O"} else 1.25
            if d < lim: ax.plot([P[i, 0], P[j, 0]], [P[i, 1], P[j, 1]], "-", color="#8a8a8a", lw=3.2, zorder=1)
            elif k != {"H"} and "H" in k and d < 2.0: ax.plot([P[i, 0], P[j, 0]], [P[i, 1], P[j, 1]], "--", color="#c0873a", lw=1.8, zorder=1)
    for i in sorted(range(n), key=lambda q: P[q, 2]):
        sc = 1.0 + 0.25 * P[i, 2]
        ax.add_patch(plt.Circle((P[i, 0], P[i, 1]), rad[S[i]] * sc * size, fc=colr[S[i]], ec="#333", lw=0.8, zorder=2 + int(P[i, 2] * 4)))
    ax.set_aspect("equal"); ax.set_xlim(-2.2, 3.0); ax.set_ylim(-1.6, 2.6); ax.axis("off")
    if title: ax.set_title(title, fontsize=9)


In [ ]:
#@title 6. Skala on the GPU: smoke test and timing (stop here if this fails). PART A
SKALA_STAND_IN = False   #@param {type:"boolean"}
# True = dry run: PBE/STO-3G on the CPU replaces Skala so the whole notebook can be tested in minutes. NEVER report numbers from a stand-in run.
if SKALA_STAND_IN:
    BASIS = "sto-3g"; sk = fpes.DFT("pbe", "sto-3g", gpu=False); where = "STAND-IN (PBE/STO-3G CPU)"; print("*** DRY RUN: PBE/STO-3G stands in for Skala. Do not report these numbers. ***")
else:
    try:
        sk = fpes.Skala(BASIS, gpu=bool(GPU))
        where = "GPU" if GPU else "CPU"
    except Exception as e:
        print("GPU Skala unavailable, falling back to CPU:", repr(e)[:160]); sk = fpes.Skala(BASIS, gpu=False); where = "CPU"
i0, j0 = int(np.argmin(np.abs(theta - 30))), int(np.argmin(np.abs(r - 1.4)))          # near the enol minimum
for k in range(2):
    sk.reset_guess(); t = fpes.hcn_geometry(theta[i0], r[j0]); out = sk(t)
    print(f"[{where}] test {k+1}: E = {out['E']:.6f} Eh, converged {out['converged']}, {out['niter']} iterations, {out['seconds']:.1f} s")
assert np.isfinite(out["E"]), "Skala returned NaN: check the install cell output and the GPU runtime"
SEC_PER_POINT = out["seconds"]
print(f"\nabout {SEC_PER_POINT:.1f} s per point -> {N_THETA*N_R} grid points would take {N_THETA*N_R*SEC_PER_POINT/60:.0f} min; the whole Part B uses stored values, so it needs no more Skala time")
if not out["converged"]: print("NOTE: the test point did not converge. The oracle retries once and then excludes failed points; check the SCF-failure counts below.")

# a second, harder probe: near the saddle (stretched, strained geometry), because SCF trouble would show here first
isad = int(np.argmin(np.abs(theta - 90))); jsad = int(np.argmin(np.abs(r - 1.0)))
sk.reset_guess(); o2 = sk(fpes.hcn_geometry(theta[isad], r[jsad]))
print(f"[{where}] saddle-region probe (theta {theta[isad]:.0f}, r {r[jsad]:.2f}): E = {o2['E']:.6f} Eh, converged {o2['converged']}, {o2['niter']} iterations, {o2['seconds']:.1f} s")
print("Projected whole-grid time at the slower of the two probes:", f"{N_THETA*N_R*max(out['seconds'], o2['seconds'])/60:.0f} min (every point is cached, so this bounds the total Skala time of the notebook)")


In [ ]:
#@title 7. STEP 1 (GPU, about 5 min) · The hydrogen in flight: Skala electron density along the cheapest path, and two bond meters
from pyscf import gto, dft
from pyscf.dft import numint
BOHR_ = fpes.BOHR
IC1, IC2, IO, IH = 0, 1, 2, 3      # atom order of the geometry: C1, C2, O, migrating H, then three spectator H
def build_mf(atoms):
    mol = gto.M(atom=fpes.atoms_to_pyscf(atoms), basis=sk.basis, unit="Angstrom", verbose=0)
    if hasattr(sk, "SkalaKS"):
        try: mf = sk.SkalaKS(mol, xc=sk.xc, with_dftd3=sk.with_dftd3)
        except TypeError: mf = sk.SkalaKS(mol, xc=sk.xc)
    else:
        mf = dft.RKS(mol); mf.xc = sk.xc; mf.grids.level = sk.grids_level
    mf.conv_tol = 1e-9; mf.max_cycle = 100
    return mol, mf
def density_on_plane(atoms, window, n=(150, 120)):
    mol, mf = build_mf(atoms); e = mf.kernel(); ok = bool(mf.converged)
    dm = mf.make_rdm1(); dm = dm.get() if hasattr(dm, "get") else np.asarray(dm)
    x = np.linspace(window[0], window[1], n[0]); y = np.linspace(window[2], window[3], n[1]); X, Y = np.meshgrid(x, y)
    pts = np.column_stack([X.ravel(), Y.ravel(), np.zeros(X.size)]) / BOHR_
    rho = numint.eval_rho(mol, numint.eval_ao(mol, pts), dm).reshape(X.shape)
    P = np.array([c for _, c in atoms])
    def meter(a, b):      # lowest density on the straight line between two nuclei, away from the nuclei themselves (a bond-strength proxy)
        t = np.linspace(0.15, 0.85, 45)[:, None]; seg = P[a][None] + t * (P[b] - P[a])[None]
        return float(numint.eval_rho(mol, numint.eval_ao(mol, seg / BOHR_), dm).min())
    return dict(E=float(e), ok=ok, rho=rho, x=x, y=y, m_OH=meter(IH, IO), m_CH=meter(IH, IC1))
def q_of(atoms):
    P = np.array([c for _, c in atoms]); return float(np.linalg.norm(P[IH] - P[IO]) - np.linalg.norm(P[IH] - P[IC1]))
if RUN_DENSITY:
    t_s1 = time.time()
    path = fpes.bottleneck_path(E_REF, theta)
    qs = np.array([q_of(fpes.hcn_geometry(theta[i], r[j])) for i, j in path])
    sad_node = max(path, key=lambda n: E_REF[n]); k_sad = path.index(sad_node)
    targets = np.linspace(qs.min(), qs.max(), N_DENS); targets[np.argmin(np.abs(targets - qs[k_sad]))] = qs[k_sad]
    pick = sorted({int(np.argmin(np.abs(qs - t))) for t in targets}, key=lambda k_: qs[k_])      # ordered by q, not by path position
    nodes = [path[k] for k in pick]
    geoms = [fpes.hcn_geometry(theta[i], r[j]) for i, j in nodes]
    allP = np.vstack([[c for _, c in g] for g in geoms]); win = (allP[:, 0].min() - 1.3, allP[:, 0].max() + 1.3, allP[:, 1].min() - 1.3, allP[:, 1].max() + 1.3)
    print(f"{len(nodes)} geometries along the cheapest path (migration coordinate q = d(O-H) - d(C1-H) from {qs.min():+.2f} to {qs.max():+.2f} A); saddle node is frame {nodes.index(sad_node)+1}")
    DENS = []
    for n_, ((i, j), g) in enumerate(zip(nodes, geoms)):
        sk.reset_guess() if hasattr(sk, "reset_guess") else None
        d_ = density_on_plane(g, win); d_.update(node=(int(i), int(j)), q=q_of(g), atoms=g, is_saddle=((i, j) == sad_node)); DENS.append(d_)
        print(f"  frame {n_+1}/{len(nodes)}  theta {theta[i]:.0f} r {r[j]:.2f}  q {d_['q']:+.2f}  E {d_['E']:.5f} Eh  converged {d_['ok']}  rho(O-H) {d_['m_OH']:.3f}  rho(C-H) {d_['m_CH']:.3f}")
    e0 = min(d_["E"] for d_ in DENS)      # energies relative to the lowest of these frames
    # --- the filmstrip ---
    ncol = 3; nrow = int(np.ceil(len(DENS) / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(4.0 * ncol, 3.3 * nrow), constrained_layout=True)
    axs = np.atleast_1d(axs).ravel(); lev = np.linspace(-2.6, 0.6, 29)
    for ax, d_ in zip(axs, DENS):
        cs = ax.contourf(d_["x"], d_["y"], np.log10(np.clip(d_["rho"], 10 ** -2.6, None)), levels=lev, cmap="magma", extend="both")
        P = np.array([c for _, c in d_["atoms"]]); S = [s for s, _ in d_["atoms"]]
        for a_, b_ in ((IH, IO), (IH, IC1)):
            ax.plot(P[[a_, b_], 0], P[[a_, b_], 1], ":", color="white", lw=0.8, alpha=0.7)
        for p_, s_ in zip(P, S): ax.plot(p_[0], p_[1], "o", ms=8 if s_ != "H" else 5, mfc={"C": "#9aa0a6", "O": "#ff5a5a", "H": "white"}[s_], mec="black", mew=0.6)
        ax.plot(P[IH, 0], P[IH, 1], "o", ms=9, mfc="none", mec="#00e5ff", mew=1.6)
        ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(("SADDLE · " if d_["is_saddle"] else "") + f"q {d_['q']:+.2f} Å · {(d_['E']-e0)*KCAL:+.0f} kcal/mol", fontsize=9, color="#d62828" if d_["is_saddle"] else "black")
    for ax in axs[len(DENS):]: ax.axis("off")
    fig.colorbar(cs, ax=axs.tolist(), shrink=0.7, label="log10 electron density (e/bohr³), Skala SCF")
    fig.suptitle("The hydrogen in flight (cyan ring): electron density in the molecular plane, enol → saddle → keto", fontsize=11)
    fig.savefig(f"{WORKDIR}/figures/fig_density_filmstrip.png", dpi=200); plt.show()
    # --- the meters ---
    q_ = np.array([d_["q"] for d_ in DENS]); mo = np.array([d_["m_OH"] for d_ in DENS]); mc = np.array([d_["m_CH"] for d_ in DENS])
    cross = None
    dd = mo - mc
    for a_ in range(len(q_) - 1):
        if dd[a_] > 0 >= dd[a_ + 1]: cross = q_[a_] + (q_[a_ + 1] - q_[a_]) * dd[a_] / (dd[a_] - dd[a_ + 1]); break
    fig, axs = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
    axs[0].plot(q_, mo, "o-", color="#d62828", lw=2, label="O–H bond meter"); axs[0].plot(q_, mc, "o-", color="#2a6fdb", lw=2, label="C1–H bond meter")
    if cross is not None: axs[0].axvline(cross, color="#888", ls=":"); axs[0].text(cross, axs[0].get_ylim()[1] * 0.95, f" handover at q ≈ {cross:+.2f} Å", fontsize=8, va="top")
    axs[0].set_xlabel("migration coordinate q = d(O–H) − d(C1–H) (Å)"); axs[0].set_ylabel("lowest density between the two nuclei (e/bohr³)"); axs[0].legend(); axs[0].set_title("One bond fades while the other grows")
    axs[1].plot(q_, [(d_["E"] - e0) * KCAL for d_ in DENS], "o-", color="#222", lw=2); axs[1].set_xlabel("q (Å)"); axs[1].set_ylabel("Skala energy (kcal/mol, rigid path frames)"); axs[1].set_title("Energy of the same frames")
    fig.savefig(f"{WORKDIR}/figures/fig_density_meters.png", dpi=200); plt.show()
    np.savez(f"{WORKDIR}/data/density_frames.npz", q=q_, E=[d_["E"] for d_ in DENS], m_OH=mo, m_CH=mc, nodes=np.array(nodes), rho=np.array([d_["rho"] for d_ in DENS]), x=DENS[0]["x"], y=DENS[0]["y"])
    print(f"step 1 took {(time.time()-t_s1)/60:.1f} min; handover of the bond meters at q = {cross if cross is None else round(cross,2)}")
else:
    DENS = None; print("step 1 skipped (RUN_DENSITY is off)")


In [ ]:
#@title 8. STEP 2 (GPU for the Skala part, about 10 min) · Relaxed truth check: xTB relaxes every atom along the jump, Skala re-computes the energies
# What this is: a path of fully relaxed geometries (xTB forces, a stiff spring on q = d(O-H) - d(C1-H), scanned from both wells),
# then Skala energies at those geometries. It is NOT a Skala transition-state search; the caption on the poster says so.
from scipy.optimize import minimize
from tblite.interface import Calculator
ZNUM = {"H": 1, "C": 6, "N": 7, "O": 8}
SYMS = [s for s, _ in fpes.hcn_geometry(30, 1.4)]; NUMS = np.array([ZNUM[s] for s in SYMS])
def xtb_eg(x):
    """Energy (Eh) and gradient (Eh per Angstrom). If the SCF does not converge, retry with stronger damping and a hotter electronic temperature."""
    last = None
    for opts in ({}, {"mixer-damping": 0.1, "max-iter": 500}, {"mixer-damping": 0.05, "max-iter": 1000, "temperature": 1500.0}):
        try:
            c = Calculator("GFN2-xTB", NUMS, x.reshape(-1, 3) / BOHR_); c.set("verbosity", 0)
            for k_, v_ in opts.items(): c.set(k_, v_)
            res = c.singlepoint(); return float(res.get("energy")), np.array(res.get("gradient")).ravel() / BOHR_
        except Exception as ex: last = ex
    raise last
def q_x(x):
    P = x.reshape(-1, 3); return float(np.linalg.norm(P[IH] - P[IO]) - np.linalg.norm(P[IH] - P[IC1]))
def pen_fun(x, q0, K):
    e, g = xtb_eg(x)
    if q0 is None: return e, g
    P = x.reshape(-1, 3); a = P[IH] - P[IO]; b = P[IH] - P[IC1]; na, nb = np.linalg.norm(a), np.linalg.norm(b); dq = na - nb - q0
    gq = np.zeros_like(P); gq[IH] = a / na - b / nb; gq[IO] = -a / na; gq[IC1] = b / nb
    return e + 0.5 * K * dq * dq, g + K * dq * gq.ravel()
def relax(x0, q0=None, K=3.0):
    r_ = minimize(pen_fun, x0, args=(q0, K), jac=True, method="L-BFGS-B", options=dict(maxiter=600, gtol=2e-4, ftol=1e-13))
    return r_.x
def as_atoms(x): return [(s, tuple(p)) for s, p in zip(SYMS, x.reshape(-1, 3))]
def kabsch_to(x, ref, idx=(IC1, IC2, IO)):
    P = x.reshape(-1, 3); Q = ref.reshape(-1, 3); a = P[list(idx)] - P[list(idx)].mean(0); b = Q[list(idx)] - Q[list(idx)].mean(0)
    U, _, Vt = np.linalg.svd(a.T @ b); d_ = np.sign(np.linalg.det(U @ Vt)); R = U @ np.diag([1, 1, d_]) @ Vt
    return ((P - P[list(idx)].mean(0)) @ R + Q[list(idx)].mean(0)).ravel()
if RUN_RELAXED:
    t_s2 = time.time()
    x_en = relax(np.array([c for _, c in fpes.hcn_geometry(theta[EN_MIN[0]], r[EN_MIN[1]])]).ravel())
    x_ke = relax(np.array([c for _, c in fpes.hcn_geometry(theta[KE_MIN[0]], r[KE_MIN[1]])]).ravel())
    q_en, q_ke = q_x(x_en), q_x(x_ke); e_en, e_ke = xtb_eg(x_en)[0], xtb_eg(x_ke)[0]
    print(f"relaxed enol: q {q_en:+.2f}, relaxed keto: q {q_ke:+.2f}; xTB keto - enol = {(e_ke-e_en)*KCAL:+.1f} kcal/mol (relaxed)")
    Q = np.linspace(q_en, q_ke, int(round((q_ke - q_en) / N_RELAX_STEP)) + 1)
    def scan(x_start, qlist):
        out, x_ = [], x_start.copy()
        for q0 in qlist:
            x_ = relax(x_, q0); out.append((xtb_eg(x_)[0], x_.copy()))
        return out
    fwd = scan(x_en, Q); rev = scan(x_ke, Q[::-1])[::-1]
    # Scanning from one well drags the other bonds along a branch that is only good on its own side of the pass. The lowest path uses the forward branch
    # before the two branches cross and the reverse branch after; the top of that path sits at the crossing, so we locate it by bisection on q.
    EF_ = np.array([f[0] for f in fwd]); ER_ = np.array([g[0] for g in rev])
    # candidate switch points: forward branch up to k, reverse branch from k+1 on; pick the k whose path has the LOWEST top (a spurious early crossing would leave a huge jump)
    tops = [max(EF_[:k + 1].max(), ER_[k + 1:].max()) for k in range(1, len(Q) - 2)]; kc = 1 + int(np.argmin(tops)) if len(tops) else None
    if kc is not None:
        lo, hi = Q[kc], Q[kc + 1]; xf, xr = fwd[kc][1], rev[kc + 1][1]
        for _ in range(7):
            qm = 0.5 * (lo + hi); xf2, xr2 = relax(xf, qm), relax(xr, qm)
            if xtb_eg(xf2)[0] <= xtb_eg(xr2)[0]: lo, xf = qm, xf2
            else: hi, xr = qm, xr2
        ef_, er_ = xtb_eg(xf)[0], xtb_eg(xr)[0]; xc = xf if ef_ >= er_ else xr; q_cross = 0.5 * (lo + hi); gap_cross = abs(ef_ - er_) * KCAL
        X = [x_en] + [f[1] for f in fwd[1:kc + 1]] + [xc] + [g[1] for g in rev[kc + 1:-1]] + [x_ke]
        print(f"the forward and reverse branches cross at q = {q_cross:+.3f}; after bisection they differ by {gap_cross:.2f} kcal/mol there (the lowest path switches branch at this point)")
    else:
        q_cross, gap_cross = float("nan"), float("nan")
        X = [x_en] + [f[1] if f[0] <= g[0] else g[1] for f, g in zip(fwd[1:-1], rev[1:-1])] + [x_ke]
        print("no crossing of the two branches found; the lower of the two at each q is used")
    X = [kabsch_to(x, X[0]) for x in X]; QS = np.array([q_x(x) for x in X])
    EX = np.array([xtb_eg(x)[0] for x in X]); hyst = gap_cross
    print(f"{len(X)} relaxed frames from enol to keto")
    _step = float(np.max(np.abs(np.diff((EX - EX[0]) * KCAL)))); print(f"largest xTB energy step between neighbouring frames: {_step:.1f} kcal/mol")
    if _step > 25: print("WARNING: the path has a jump (two branches were joined at the wrong place). Do not quote the relaxed barrier; send the printed lines for review.")
    rel_xtb = (EX - EX[0]) * KCAL
    # largest geometric jump between neighbouring frames
    jumps = [float(np.sqrt(np.mean((X[a_+1].reshape(-1, 3) - X[a_].reshape(-1, 3)) ** 2))) for a_ in range(len(X) - 1)]
    print(f"largest frame-to-frame RMS jump {max(jumps):.2f} A (frame {int(np.argmax(jumps))+1} to {int(np.argmax(jumps))+2}); a smooth path keeps this below about 0.3 A")
    ES, SKOK = [], []
    for n_, x in enumerate(X):
        sk.reset_guess() if hasattr(sk, "reset_guess") else None
        o_ = sk(as_atoms(x)); ES.append(o_["E"]); SKOK.append(o_["converged"])
        print(f"  Skala frame {n_+1}/{len(X)}  q {QS[n_]:+.2f}  E {o_['E']:.5f}  converged {o_['converged']}  {o_['seconds']:.1f} s")
    ES = np.array(ES); rel_sk = (ES - ES[0]) * KCAL
    kmax = int(np.nanargmax(rel_sk)); peak = rel_sk[kmax]
    if 0 < kmax < len(X) - 1:
        y0, y1, y2 = rel_sk[kmax-1:kmax+2]; den = (y2 - 2 * y1 + y0); peak = y1 - (y2 - y0) ** 2 / (8 * den) if den < 0 else y1
    RELAX = dict(q=QS, rel_xtb=rel_xtb, rel_skala=rel_sk, barrier_xtb_relaxed=float(rel_xtb.max()), barrier_skala_relaxed=float(rel_sk[kmax]), barrier_skala_relaxed_refined=float(peak),
                 kmax=kmax, keto_minus_enol_skala=float(rel_sk[-1]), rigid_skala=float(REF_BARRIER), rigid_xtb=float(bx["barrier_kcal"]), q_cross=float(q_cross), gap_at_crossing=float(gap_cross), max_jump=max(jumps),
                 all_converged=bool(all(SKOK)), geoms=[a_.tolist() for a_ in X])
    print("\n          rigid grid     relaxed path (xTB geometries)\n"
          f"xTB       {RELAX['rigid_xtb']:8.1f}     {RELAX['barrier_xtb_relaxed']:8.1f}   kcal/mol\n"
          f"Skala     {RELAX['rigid_skala']:8.1f}     {RELAX['barrier_skala_relaxed']:8.1f}  (parabola peak {peak:.1f})   kcal/mol;  literature about 55 (from memory, unverified)\n"
          f"keto - enol, Skala on the relaxed path: {RELAX['keto_minus_enol_skala']:+.1f} kcal/mol (literature about -10, from memory)")
    with open(f"{WORKDIR}/data/relaxed_path.xyz", "w") as fh:
        for n_, x in enumerate(X):
            fh.write(f"{len(SYMS)}\nframe {n_} q={QS[n_]:.3f} Skala_rel={rel_sk[n_]:.2f} kcal/mol\n")
            for s_, p_ in zip(SYMS, x.reshape(-1, 3)): fh.write(f"{s_} {p_[0]:.5f} {p_[1]:.5f} {p_[2]:.5f}\n")
    json.dump({k: (v.tolist() if hasattr(v, "tolist") else v) for k, v in RELAX.items() if k != "geoms"}, open(f"{WORKDIR}/data/relaxed_summary.json", "w"), indent=1)
    # --- figure: molecules along the path and the two energy traces ---
    show = sorted({0, len(X) // 4, max(kmax - 2, 0), kmax, min(kmax + 2, len(X) - 1), 3 * len(X) // 4, len(X) - 1})
    fig = plt.figure(figsize=(13, 7), constrained_layout=True); gs = fig.add_gridspec(2, len(show), height_ratios=[1, 1.5])
    for c_, k in enumerate(show):
        draw_mol(fig.add_subplot(gs[0, c_]), as_atoms(X[k]), f"frame {k+1} · q {QS[k]:+.2f}")
    ax = fig.add_subplot(gs[1, :])
    ax.plot(QS, rel_xtb, "s--", color="#777", label=f"xTB, relaxed path (barrier {RELAX['barrier_xtb_relaxed']:.1f})")
    ax.plot(QS, rel_sk, "o-", color="#D97A12", lw=2.2, label=f"Skala energies on the same geometries (barrier {RELAX['barrier_skala_relaxed']:.1f})")
    ax.axhline(REF_BARRIER, color="#2a6fdb", ls=":", label=f"rigid-grid Skala barrier {REF_BARRIER:.1f} (NB10)")
    ax.axhline(55, color="#888", ls="-.", lw=1, label="literature about 55 (from memory, unverified)")
    ax.set_xlabel("q = d(O–H) − d(C1–H) (Å)"); ax.set_ylabel("energy above relaxed enol (kcal/mol)"); ax.legend(fontsize=8, loc="upper center")
    ax.set_title("Every atom relaxed along the hydrogen jump (xTB geometries; Skala energies are single points, not a Skala transition-state search)", fontsize=9)
    fig.savefig(f"{WORKDIR}/figures/fig_relaxed_path.png", dpi=200); plt.show()
    print(f"step 2 took {(time.time()-t_s2)/60:.1f} min")
else:
    RELAX = None; print("step 2 skipped (RUN_RELAXED is off)")


In [ ]:
#@title 9. STEP 3 (GPU, time boxed to G1_MAX_MIN minutes, resumable) · The microscope: Skala at finer spacing around the pass and the enol well
# Rerun this cell after a disconnect: finished points are read from Drive, only missing ones are computed.
import heapq
def lin(lo, hi, step): return np.linspace(lo, hi, int(round((hi - lo) / step)) + 1)
thA, rA = lin(40, 140, G1_THETA_STEP), lin(0.7, 1.7, G1_R_STEP)     # patch A: the pass
thB, rB = lin(0, 50, G1_THETA_STEP), lin(1.0, 1.7, G1_R_STEP)       # patch B: the enol well (a better reference for the barrier)
nA, nB = len(thA) * len(rA), len(thB) * len(rB)
print(f"patch A (pass): {len(thA)} x {len(rA)} = {nA} points; patch B (enol well): {len(thB)} x {len(rB)} = {nB} points; at {SEC_NB10:.1f} s per point that is about {(nA+nB)*SEC_NB10/60:.0f} min")
def pass_height(EA):
    """Lowest possible maximum along any 4-connected path from the left edge (theta min) to the right edge (theta max) of a patch."""
    nt_, nr_ = EA.shape; best = np.full((nt_, nr_), np.inf); pq = []
    for j in range(nr_):
        best[0, j] = EA[0, j]; heapq.heappush(pq, (EA[0, j], 0, j))
    prev = {}
    while pq:
        c, i, j = heapq.heappop(pq)
        if c > best[i, j]: continue
        for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            ni, nj = i + di, j + dj
            if 0 <= ni < nt_ and 0 <= nj < nr_ and np.isfinite(EA[ni, nj]):
                nc = max(c, EA[ni, nj])
                if nc < best[ni, nj]: best[ni, nj] = nc; prev[(ni, nj)] = (i, j); heapq.heappush(pq, (nc, ni, nj))
    jend = int(np.argmin(best[-1])); node = (nt_ - 1, jend); path_ = [node]
    while node in prev: node = prev[node]; path_.append(node)
    path_ = path_[::-1]; top = max(path_, key=lambda n: EA[n])
    return float(best[-1, jend]), top, path_
if RUN_MICRO:
    t_s3 = time.time(); pA, pB = f"{WORKDIR}/skala_patchA.npz", f"{WORKDIR}/skala_patchB.npz"
    resA = fpes.compute_surface(sk, thA, rA, pA, log_every=25, max_minutes=0.65 * G1_MAX_MIN)
    left = G1_MAX_MIN - (time.time() - t_s3) / 60
    resB = fpes.compute_surface(sk, thB, rB, pB, log_every=25, max_minutes=max(left, 1.0)) if left > 0.5 or np.isnan(resA["E"]).sum() == 0 else None
    if resB is None: print("the time box is used up before patch B started; rerun this cell to continue (patch A is saved)")
    EA = resA["E"]; EB = resB["E"] if resB is not None else np.full((len(thB), len(rB)), np.nan)
    doneA, doneB = int((~np.isnan(EA)).sum()), int((~np.isnan(EB)).sum())
    print(f"patch A {doneA}/{nA} done, patch B {doneB}/{nB} done")
    MICRO = None
    if doneA == nA:
        Eref0 = min(E_REF[EN_MIN], np.nanmin(EB) if doneB else np.inf)
        if SKALA_STAND_IN: Eref0 = np.nanmin(EB) if doneB else np.nanmin(EA)      # dry run only: the stand-in energies are not on the stored Skala scale           # the enol-well reference: lowest of the coarse minimum and the fine patch
        ph, top, ppath = pass_height(EA)
        fine_barrier = (ph - Eref0) * KCAL
        MICRO = dict(fine_pass_barrier=float(fine_barrier), coarse_barrier=float(REF_BARRIER), pass_theta=float(thA[top[0]]), pass_r=float(rA[top[1]]),
                     enol_ref_from_fine_patch=bool(doneB and np.nanmin(EB) < E_REF[EN_MIN]), n_fine_points=int(doneA + doneB),
                     coarse_saddle_theta=float(theta[SAD[0]]), coarse_saddle_r=float(r[SAD[1]]),
                     enol_ref_shift_kcal=float((E_REF[EN_MIN] - Eref0) * KCAL))
        print(f"\nFINE PASS: highest point on the lowest path across patch A = {fine_barrier:.2f} kcal/mol above the enol reference, at theta {MICRO['pass_theta']:.0f}, r {MICRO['pass_r']:.2f}"
              f"\nCOARSE (NB10, 19 x 13 grid): {REF_BARRIER:.2f} kcal/mol at theta {theta[SAD[0]]:.0f}, r {r[SAD[1]]:.2f}; the enol reference moved by {MICRO['enol_ref_shift_kcal']:.2f} kcal/mol with the finer well patch")
        json.dump(MICRO, open(f"{WORKDIR}/data/microscope_summary.json", "w"), indent=1)
        # --- zoom figure: coarse map, fine pass patch, fine well patch ---
        vmax = 1.5 * WALL
        fig, axs = plt.subplots(1, 3, figsize=(16, 4.6), constrained_layout=True, gridspec_kw=dict(width_ratios=[1.2, 1, 1]))
        Z = (E_REF - Eref0) * KCAL
        cs = axs[0].contourf(theta, r, Z.T, levels=np.linspace(0, vmax, 25), cmap=CMAP, extend="max")
        for (t0_, t1_, r0_, r1_, c_) in ((40, 140, 0.7, 1.7, "white"), (0, 50, 1.0, 1.7, "#7fe7ff")):
            axs[0].add_patch(plt.Rectangle((t0_, r0_), t1_ - t0_, r1_ - r0_, fill=False, ec=c_, lw=1.8))
        axs[0].plot(theta[SAD[0]], r[SAD[1]], "r*", ms=14); axs[0].set_title(f"coarse 19 × 13 (NB10): barrier {REF_BARRIER:.1f}"); axs[0].set_xlabel("θ"); axs[0].set_ylabel("r (Å)")
        ZA = (EA - Eref0) * KCAL
        pm = axs[1].pcolormesh(thA, rA, ZA.T, cmap=CMAP, vmin=0, vmax=vmax, shading="nearest")
        axs[1].contour(thA, rA, ZA.T, levels=[fine_barrier], colors="white", linewidths=1.2)
        axs[1].plot([thA[n[0]] for n in ppath], [rA[n[1]] for n in ppath], "-", color="white", lw=1.2, alpha=0.8)
        axs[1].plot(thA[top[0]], rA[top[1]], "r*", ms=14); axs[1].plot(theta[SAD[0]], r[SAD[1]], "kx", ms=10, mew=2)
        axs[1].set_title(f"microscope on the pass: {fine_barrier:.1f} (red star), coarse pick (black x)"); axs[1].set_xlabel("θ")
        if doneB:
            ZB = (EB - Eref0) * KCAL; axs[2].pcolormesh(thB, rB, np.clip(ZB.T, 0, 40), cmap=CMAP, vmin=0, vmax=40, shading="nearest")
            iB = np.unravel_index(np.nanargmin(EB), EB.shape); axs[2].plot(thB[iB[0]], rB[iB[1]], "w*", ms=12)
            axs[2].plot(theta[EN_MIN[0]], r[EN_MIN[1]], "kx", ms=10, mew=2); axs[2].set_title("microscope on the enol well (colour 0–40)"); axs[2].set_xlabel("θ")
        fig.colorbar(cs, ax=axs[:2], shrink=0.8, label="E above enol reference (kcal/mol)")
        fig.savefig(f"{WORKDIR}/figures/fig_microscope_zoom.png", dpi=200); plt.show()
        # --- 3D mountain pass ---
        from mpl_toolkits.mplot3d import Axes3D  # noqa
        fig = plt.figure(figsize=(8.5, 6.5)); ax3 = fig.add_subplot(111, projection="3d"); TT, RR = np.meshgrid(thA, rA, indexing="ij")
        ax3.plot_surface(TT, RR, np.clip(ZA, 0, 130), cmap=CMAP, edgecolor="none", alpha=0.92, rcount=60, ccount=60)
        ax3.plot([thA[n[0]] for n in ppath], [rA[n[1]] for n in ppath], [ZA[n] + 1.5 for n in ppath], "-", color="white", lw=2.2)
        ax3.scatter([thA[top[0]]], [rA[top[1]]], [ZA[top] + 2], s=90, c="red", marker="*")
        ci, cj = np.meshgrid(range(len(theta)), range(len(r)), indexing="ij"); msk = (theta[ci] >= 40) & (theta[ci] <= 140) & (r[cj] >= 0.7) & (r[cj] <= 1.7)
        ax3.scatter(theta[ci][msk], r[cj][msk], np.clip(Z[msk], 0, 130) + 1, s=10, c="k", depthshade=False)
        ax3.set_xlabel("θ"); ax3.set_ylabel("r (Å)"); ax3.set_zlabel("kcal/mol"); ax3.view_init(elev=28, azim=-62); ax3.set_title("The pass seen from the enol side (black dots = the 19 × 13 grid points)")
        fig.savefig(f"{WORKDIR}/figures/fig_microscope_3d.png", dpi=200); plt.show()
    else:
        print("patch A is not complete yet: rerun this cell (it resumes).")
    print(f"step 3 took {(time.time()-t_s3)/60:.1f} min")
else:
    MICRO = None; EA = EB = None; print("step 3 skipped (RUN_MICRO is off)")


In [ ]:
#@title 10. STEP 4 (CPU, about 3 min) · The replay: how the agent's map grows, pick by pick, next to the best classical rule
from PIL import Image
t_s4 = time.time()
run_a = [x for x in agent_runs["menu|U1_barrier"] if x["seed"] == SEED_SHOW][0]
run_c = [x for x in classical["saddle_exploit+delta"] if x["seed"] == SEED_SHOW][0]
idx_a = [tuple(int(v) for v in p) for p in run_a["idx"]]; idx_c = [tuple(int(v) for v in p) for p in run_c["idx"]]
K_END_A = len(idx_a); K_ALL = list(range(4, max(len(idx_c), K_END_A) + 1))
Zmin = E_REF.min(); vmax_map = 1.5 * WALL
def est_of(idx_k):
    m, _ = run_mean(idx_k); b = fpes.bottleneck_barrier(m, theta); return m, b
def panel_map(ax, idx_k, title, why=None, stopped=False):
    m, b = est_of(idx_k)
    ax.contourf(theta, r, ((m - Zmin) * KCAL).T, levels=np.linspace(0, vmax_map, 25), cmap=CMAP, extend="max")
    ax.contour(theta, r, ((E_REF - Zmin) * KCAL).T, levels=[(E_REF[SAD] - Zmin) * KCAL], colors="white", linewidths=0.8, linestyles="--")
    ax.plot(theta[SAD[0]], r[SAD[1]], "*", color="#ffd400", ms=13, mec="black")
    for n_, (i, j) in enumerate(idx_k): ax.plot(theta[i], r[j], "o", ms=11, mfc="white", mec="black"); ax.text(theta[i], r[j], str(n_ + 1), ha="center", va="center", fontsize=6.5)
    if b["saddle"] is not None: ax.plot(theta[b["saddle"][0]], r[b["saddle"][1]], "x", color="red", ms=11, mew=2.5)
    err = b["barrier_kcal"] - REF_BARRIER
    ax.set_title(f"{title}\n{len(idx_k)} Skala calls · barrier {b['barrier_kcal']:.1f} (error {err:+.1f})" + ("   [STOPPED]" if stopped else ""), fontsize=9)
    ax.set_xlabel("θ"); ax.set_ylabel("r (Å)")
    if why: ax.text(0.5, -0.34, "\n".join(textwrap.wrap("agent: " + why, 70)[:3]), transform=ax.transAxes, ha="center", va="top", fontsize=7.5, style="italic")
    return err
def frame_img(k):
    ka = min(k, K_END_A); kc = k
    fig = plt.figure(figsize=(11.5, 6.4)); gs = fig.add_gridspec(2, 2, height_ratios=[3.2, 1.1], hspace=0.95, wspace=0.22)
    ax1, ax2, ax3 = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[1, :])
    why = run_a["decisions"][ka - 5]["why"] if ka >= 5 and ka - 5 < len(run_a["decisions"]) else "first four points: a fixed starting design, the same for every method"
    panel_map(ax1, idx_a[:ka], "MENU AGENT (Skala picks numbered)", why, stopped=(k >= K_END_A))
    panel_map(ax2, idx_c[:kc], "BEST CLASSICAL RULE (saddle exploit + Δ-learning)")
    for hist, col, lab, kk in ((run_a["hist"], COL["menu"], "menu agent", ka), (run_c["hist"], COL["classical"], "best classical rule", kc)):
        xs = [h["calls"] for h in hist if h["calls"] <= kk]; ys = [abs(h["error"]) for h in hist if h["calls"] <= kk]
        ax3.plot(xs, np.maximum(ys, 0.02), "o-", color=col, lw=2, label=lab, ms=4)
    ax3.axhline(1, color="k", ls=":"); ax3.set_yscale("log"); ax3.set_xlim(3.5, K_ALL[-1] + 0.5); ax3.set_ylim(0.01, 40)
    ax3.set_xlabel("Skala calls"); ax3.set_ylabel("|barrier error| (kcal/mol)"); ax3.legend(fontsize=8, loc="upper right")
    ax3.set_title(f"Start {SEED_SHOW}: yellow star = true pass, red × = the method's current estimate, dashed white = true pass height", fontsize=8)
    fig.canvas.draw(); im = Image.fromarray(np.asarray(fig.canvas.buffer_rgba())[:, :, :3].copy()); plt.close(fig); return im
frames = [frame_img(k) for k in K_ALL]
frames += [frames[-1]] * 3
frames[0].save(f"{WORKDIR}/figures/replay_menu_vs_classical.gif", save_all=True, append_images=frames[1:], duration=[900] * (len(frames) - 3) + [2500] * 3, loop=0)
frames[min(len(frames) - 4, K_END_A - 4)].save(f"{WORKDIR}/figures/fig_replay_agent_stops.png")
frames[-1].save(f"{WORKDIR}/figures/fig_replay_final.png")
from IPython.display import Image as IPImage, display
display(IPImage(filename=f"{WORKDIR}/figures/replay_menu_vs_classical.gif"))
print(f"step 4: {len(frames)} frames; the menu agent stopped itself at {K_END_A} calls (error {run_a['hist'][-1]['error']:+.2f}); the classical rule ran to {len(idx_c)} (error {run_c['hist'][-1]['error']:+.2f}, {abs(run_c['hist'][K_END_A-4]['error']):.2f} at {K_END_A} calls); took {(time.time()-t_s4)/60:.1f} min")


In [ ]:
#@title 11. STEP 5 (CPU, about 3 min) · The fair-fight control: a hand-made "pass, then both wells, then refine" rule
# Why: in NB10 the menu agent won because it measured the pass AND the enol well. Does a rule that simply does those two things by hand match it?
# anchor-first: initial design (same 4 points), then Skala calls at the xTB saddle node, the xTB enol-well node and the xTB keto-well node, then the best classical rule (saddle exploit + delta).
from scipy import stats
t_s5 = time.time()
def run_anchor_first(seed, budget=BUDGET, n_init=4):
    rng = np.random.default_rng(seed); orc = fpes.GridOracle(E_REF, theta, r); idx = fpes.initial_design(theta, r, n_init, rng)
    y = [orc(i, j) for i, j in idx]
    forced = [tuple(int(v) for v in bx[k_]) for k_ in ("saddle", "hcn", "hnc")]
    hist = []
    while True:
        mean, std = fpes.gp_predict(theta, r, idx, y, baseline=E_xtb)
        for p_, (i, j) in enumerate(idx): mean[i, j] = y[p_]
        est = fpes.bottleneck_barrier(mean, theta)["barrier_kcal"]; hist.append(dict(calls=len(idx), barrier=est, error=est - REF_BARRIER))
        if len(idx) >= budget: break
        nxt = None
        while forced and nxt is None:
            c_ = forced.pop(0)
            if c_ not in idx: nxt = c_
        if nxt is None: nxt = tuple(int(v) for v in fpes.choose_next("saddle_exploit", theta, r, idx, mean, std, rng=rng))
        idx.append(nxt); y.append(orc(*nxt))
    return dict(seed=seed, hist=hist, idx=[list(p) for p in idx], delta=True)
CTRL = [run_anchor_first(s) for s in range(SEEDS)]
json.dump(CTRL, open(f"{WORKDIR}/data/anchor_first_runs.json", "w"))
BIG = BUDGET + 1
def row_of(run):
    h = run["hist"]; q = stop_index(h); to1 = calls_to_tol(h, 1.0)
    return dict(seed=run["seed"], to1=BIG if to1 is None else to1, stop_calls=h[q]["calls"], stop_err=abs(h[q]["error"]))
CT = pd.DataFrame([row_of(x) for x in CTRL]).set_index("seed")
ARM_MENU, ARM_BEST = "menu", "saddle_exploit+delta+stop rule"
def tab(arm, goal): return TAB10[(TAB10.arm == arm) & (TAB10.goal == goal)].set_index("seed").sort_index()
M_, B_ = tab(ARM_MENU, "U1_barrier"), tab(ARM_BEST, "any")
def wil(a, b, col):
    sd = a.index.intersection(b.index); d = (a.loc[sd, col] - b.loc[sd, col]).values
    p = 1.0 if np.all(d == 0) else stats.wilcoxon(d, zero_method="zsplit").pvalue
    return f"median diff {np.median(d):+.1f}  (control better in {int((d<0).sum())}, tie {int((d==0).sum())}, worse {int((d>0).sum())}), p = {p:.3f}"
print("per-start table (calls to 1 kcal/mol; calls at stop; |error| at stop):")
summary = pd.DataFrame({"arm": ["menu agent", "best classical rule", "anchor-first control"],
    "to1_median": [M_.to1.median(), B_.to1.median(), CT.to1.median()], "stop_calls_median": [M_.stop_calls.median(), B_.stop_calls.median(), CT.stop_calls.median()],
    "stop_err_median": [M_.stop_err.median(), B_.stop_err.median(), CT.stop_err.median()], "stop_err_max": [M_.stop_err.max(), B_.stop_err.max(), CT.stop_err.max()]}).round(2)
print(summary.to_string(index=False))
for col, lab in (("to1", "calls to 1 kcal/mol"), ("stop_calls", "calls at stop"), ("stop_err", "error at stop")):
    print(f"\n{lab}:\n  control vs menu agent        : {wil(CT, M_, col)}\n  control vs best classical    : {wil(CT, B_, col)}")
summary.to_csv(f"{WORKDIR}/data/control_summary.csv", index=False); CT.to_csv(f"{WORKDIR}/data/anchor_first_per_start.csv")
fig, axs = plt.subplots(1, 3, figsize=(13, 4.2), constrained_layout=True); rng_ = np.random.default_rng(1)
ARMS5 = [("menu agent", M_, COL["menu"]), ("best classical\nrule", B_, COL["classical"]), ("anchor-first\ncontrol", CT, COL["control"])]
for ax, (col, lab, logy) in zip(axs, (("to1", f"calls to 1 kcal/mol ({BIG} = never)", False), ("stop_calls", "calls at stop", False), ("stop_err", "|barrier error| at stop (kcal/mol)", True))):
    for k, (nm, df, c_) in enumerate(ARMS5):
        v = df[col].values.astype(float); ax.plot(k + rng_.uniform(-0.15, 0.15, len(v)), np.maximum(v, 0.01) if logy else v, "o", color=c_, alpha=0.85, ms=7)
        ax.plot([k - 0.28, k + 0.28], [np.median(v)] * 2, "k-", lw=2.2)
    ax.set_xticks(range(3)); ax.set_xticklabels([a[0] for a in ARMS5], fontsize=9); ax.set_ylabel(lab)
    if logy: ax.set_yscale("log"); ax.axhline(1, color="k", ls=":")
fig.suptitle("Same 8 starts, same stopping rule. Black bar = median. Does a hand-made 'pass, then both wells' rule match the agent?", fontsize=10)
fig.savefig(f"{WORKDIR}/figures/fig_control_anchor_first.png", dpi=200); plt.show()
print(f"step 5 took {(time.time()-t_s5)/60:.1f} min")


In [ ]:
#@title 12. STEP 6 (CPU, about 2 min) · The poster: four panels A-D, each also saved on its own
from matplotlib.gridspec import GridSpecFromSubplotSpec as GSS
def note(ax, text): ax.text(0.5, 0.5, text, ha="center", va="center", transform=ax.transAxes, color="#888"); ax.axis("off")
def panelA(fig, spec):
    ax = fig.add_subplot(spec); Zr = (E_REF - E_REF.min()) * KCAL
    ax.contourf(theta, r, Zr.T, levels=np.linspace(0, 1.5 * WALL, 25), cmap=CMAP, extend="max")
    if MICRO is not None:
        ZA_ = (EA - E_REF.min()) * KCAL; ax.pcolormesh(thA, rA, ZA_.T, cmap=CMAP, vmin=0, vmax=1.5 * WALL, shading="nearest"); ax.add_patch(plt.Rectangle((thA[0], rA[0]), thA[-1] - thA[0], rA[-1] - rA[0], fill=False, ec="white", lw=1.4))
    idx_ = [tuple(int(v) for v in p) for p in run_a["idx"]]
    for n_, (i, j) in enumerate(idx_):
        ax.plot(theta[i], r[j], "o", ms=12, mfc="white", mec="black"); ax.text(theta[i], r[j], str(n_ + 1), ha="center", va="center", fontsize=7)
    ax.plot(theta[SAD[0]], r[SAD[1]], "*", color="#ffd400", ms=16, mec="black")
    ax.set_xlabel("θ  (0 = hydrogen on oxygen, 180 = on carbon)"); ax.set_ylabel("r (Å)")
    ax.set_title(f"A · The agent's {len(idx_)} Skala calls (numbered) find the pass (star)", fontsize=10, loc="left", pad=8); return ax
def panelB(fig, spec):
    if RELAX is None: ax = fig.add_subplot(spec); note(ax, "step 2 not run"); return ax
    g = GSS(3, 3, subplot_spec=spec, height_ratios=[0.16, 1, 1.35], hspace=0.3, wspace=0.05); tax = fig.add_subplot(g[0, :]); tax.axis("off"); tax.text(0, 0.2, "B · Every atom relaxed along the jump (xTB geometries, Skala energies)", fontsize=10, transform=tax.transAxes); X = [np.array(x) for x in RELAX["geoms"]]; k = RELAX["kmax"]
    for c_, kk in enumerate((0, k, len(X) - 1)): draw_mol(fig.add_subplot(g[1, c_]), as_atoms(X[kk]), ("enol", "top of the pass", "keto")[c_])
    ax = fig.add_subplot(g[2, :]); ax.plot(RELAX["q"], RELAX["rel_skala"], "o-", color="#D97A12", lw=2.2, label=f"relaxed path: {RELAX['barrier_skala_relaxed']:.0f}")
    ax.axhline(RELAX["rigid_skala"], color="#2a6fdb", ls=":", label=f"rigid grid: {RELAX['rigid_skala']:.0f}"); ax.axhline(55, color="#888", ls="-.", lw=1, label="literature ~55*")
    ax.set_xlabel("q = d(O–H) − d(C1–H) (Å)"); ax.set_ylabel("kcal/mol above enol"); ax.legend(fontsize=7.5); return ax
def panelC(fig, spec):
    if DENS is None: ax = fig.add_subplot(spec); note(ax, "step 1 not run"); return ax
    g = GSS(3, 3, subplot_spec=spec, height_ratios=[0.16, 1, 1.0], hspace=0.3, wspace=0.08); tax = fig.add_subplot(g[0, :]); tax.axis("off"); tax.text(0, 0.2, "C · The hydrogen hands over from oxygen to carbon (Skala density)", fontsize=10, transform=tax.transAxes); sel = [0, next((k_ for k_, d_ in enumerate(DENS) if d_["is_saddle"]), len(DENS) // 2), len(DENS) - 1]
    for c_, kk in enumerate(sel):
        ax = fig.add_subplot(g[1, c_]); d_ = DENS[kk]
        ax.contourf(d_["x"], d_["y"], np.log10(np.clip(d_["rho"], 10 ** -2.6, None)), levels=np.linspace(-2.6, 0.6, 29), cmap="magma", extend="both")
        P = np.array([c for _, c in d_["atoms"]]); ax.plot(P[IH, 0], P[IH, 1], "o", ms=9, mfc="none", mec="#00e5ff", mew=1.6); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(("enol", "saddle", "keto")[c_], fontsize=8)
    ax = fig.add_subplot(g[2, :]); q_ = [d_["q"] for d_ in DENS]; ax.plot(q_, [d_["m_OH"] for d_ in DENS], "o-", color="#d62828", label="O–H bond"); ax.plot(q_, [d_["m_CH"] for d_ in DENS], "o-", color="#2a6fdb", label="C–H bond")
    ax.set_xlabel("q (Å)"); ax.set_ylabel("bond meter"); ax.legend(fontsize=7.5); return ax
def panelD(fig, spec):
    ax = fig.add_subplot(spec); n_pts = E_REF.size
    rows = [("Scan all points", n_pts, np.nan, "#999999"),
            ("Best classical", B_.stop_calls.median(), B_.stop_err.median(), COL["classical"]),
            ("Anchor-first", CT.stop_calls.median(), CT.stop_err.median(), COL["control"]),
            ("Menu agent", M_.stop_calls.median(), M_.stop_err.median(), COL["menu"])]
    for k, (nm, n_, e_, c_) in enumerate(rows[::-1]):
        ax.barh(k, n_, color=c_); ax.text(n_ + 3, k, f"{n_:.0f} calls · {n_*SEC_NB10/60:.1f} min of Skala" + ("" if np.isnan(e_) else f" · error {e_:.2f}"), va="center", fontsize=8.5)
    ax.set_yticks(range(4)); ax.set_yticklabels([x[0] for x in rows[::-1]]); ax.set_xlim(0, n_pts * 1.85); ax.set_xlabel(f"Skala calls (time = calls × {SEC_NB10:.1f} s, NB10 L4; the agent's LLM time is not included)")
    ax.set_title("D · What it costs to find the barrier (median of 8 starts; error in kcal/mol)", fontsize=10, loc="left"); return ax
def panel_png(fn, name, size=(7, 5.2)):
    fig = plt.figure(figsize=size, constrained_layout=False); gs_ = fig.add_gridspec(1, 1, left=0.12, right=0.97, bottom=0.14, top=0.86); fn(fig, gs_[0]); fig.savefig(f"{WORKDIR}/figures/{name}", dpi=200); plt.close(fig)
for fn, nm in ((panelA, "poster_panel_A_map.png"), (panelB, "poster_panel_B_relaxed.png"), (panelC, "poster_panel_C_density.png"), (panelD, "poster_panel_D_cost.png")): panel_png(fn, nm)
fig = plt.figure(figsize=(16, 10.5)); gs = fig.add_gridspec(2, 2, left=0.055, right=0.985, bottom=0.085, top=0.88, wspace=0.28, hspace=0.34)
panelA(fig, gs[0, 0]); panelB(fig, gs[0, 1]); panelC(fig, gs[1, 0]); panelD(fig, gs[1, 1])
HEAD_N = int(round(M_.stop_calls.median()))
fig.suptitle(f"Finding the barrier with {HEAD_N} expensive calls instead of {E_REF.size}", fontsize=19, fontweight="bold", x=0.055, ha="left", y=0.965)
fig.text(0.055, 0.915, "Vinyl alcohol → acetaldehyde (hydrogen shift) with the Skala functional: an LLM agent picks which points to compute by reading a menu", fontsize=11, color="#444")
fig.text(0.055, 0.025, "Grid: 19 × 13 rigid recipe, Skala def2-SVP. B: xTB-relaxed geometries with Skala single-point energies, not a Skala transition-state search. *Literature barrier quoted from memory, unverified. Panels C, D: Skala SCF density; D medians over 8 starts, post-hoc stopping rule for non-agent arms.", fontsize=7.5, color="#555")
fig.savefig(f"{WORKDIR}/figures/POSTER_four_panels.png", dpi=220); plt.show()


In [ ]:
#@title 13. Save everything: summary, zip (also copied to Drive), download
t_ = {"nb": 11, "system": "vinyl alcohol -> acetaldehyde", "skala_basis": BASIS, "stand_in": bool(SKALA_STAND_IN), "seconds_per_skala_point_nb10": SEC_NB10,
      "coarse_barrier": float(REF_BARRIER), "microscope": MICRO, "relaxed": {k_: v_ for k_, v_ in (RELAX or {}).items() if k_ not in ("geoms", "q", "rel_xtb", "rel_skala")},
      "density_meter_handover_q": (None if DENS is None else (None if cross is None else float(cross))),
      "control_summary": summary.to_dict(orient="records"), "replay_seed": SEED_SHOW}
json.dump(t_, open(f"{WORKDIR}/summary_nb11.json", "w"), indent=1, default=float)
out = "/content/nb11_visuals" if os.path.isdir("/content") else os.path.abspath("./nb11_visuals")
shutil.rmtree(out, ignore_errors=True); os.makedirs(out)
for sub_ in ("figures", "data"): shutil.copytree(f"{WORKDIR}/{sub_}", f"{out}/{sub_}", ignore=shutil.ignore_patterns("agent_runs_powered.json", "classical_runs.json"))
shutil.copy(f"{WORKDIR}/summary_nb11.json", out)
zp = shutil.make_archive(out, "zip", out); shutil.copy(zp, f"{WORKDIR}/nb11_visuals.zip")
print("zip:", zp, f"({os.path.getsize(zp)/1e6:.1f} MB); copy on Drive:", f"{WORKDIR}/nb11_visuals.zip")
try:
    from google.colab import files; files.download(zp)
except Exception: print("download by hand from Drive (folder fruitful-pes-vinyl-alcohol-visuals)")
if SKALA_STAND_IN: print("*** DRY RUN: nothing above is a Skala result. ***")


In [ ]:
#@title 14. Release the GPU (optional; turn the switch on only after the zip is on Drive)
DISCONNECT = False   #@param {type:"boolean"}
if DISCONNECT:
    from google.colab import runtime; runtime.unassign()
else:
    print("switch is off: the runtime stays connected")


## Reading the pictures
| picture | what to say | what not to say |
|---|---|---|
| Density filmstrip + meters (step 1) | The hydrogen hands over from oxygen to carbon; the Skala density shows the bond switch near the pass. | That Skala "understands" the reaction. It is a density picture of a rigid-path frame. |
| Relaxed path (step 2) | Letting all atoms relax lowers the barrier from the rigid-grid value toward the literature range; the agent's advantage was measured on the rigid grid. | That Skala reproduces the literature barrier: these are xTB geometries with Skala energies, and the literature value is from memory. |
| Microscope (step 3) | The 19 × 13 grid is coarse; a finer Skala re-scan gives a pass height that differs by X kcal/mol, which is the size of the discretisation. | A new barrier for the reaction. |
| Replay (step 4) | One typical start, shown honestly (pick numbers and the agent's own reasons). | That every start looks like this: the per-start table is in NB10. |
| Control (step 5) | Report whatever it shows. If the hand-made rule matches the agent, the claim becomes "the agent found the right recipe by itself"; if not, "the menu's choices are not trivially reproducible". | |
| Poster | Panels A–D each make one point; D carries the cost ledger. | Speed claims in seconds: the LLM time is not on the ledger. |

**Stand-in runs are for testing only.** Never report numbers produced with `SKALA_STAND_IN = True`.
